## Instructions for use
### Setup
Running these two cells sets up GMAT and the simulation objects. Run these every time. You will need the GMAT Python API startup file downloaded for the first cell. The instructions for doing so can be found in the official GMAT download package, found on this website: https://sourceforge.net/projects/gmat/
The second cell includes default parameters for the satellite and the space environment. These parameters get reset to different values later where ever GMAT is reinitialized with these commands:
gmat.Initialize()
pdprop.PrepareInternals()
### Functions
The first five functions comprise the physics model for this simulation. Run these every time. You can modify and change these functions all you want as long as they maintain their names, input formats, and output names/formats. For example, the B-field has to have a datetime and cartesian position vector as its inputs and a cartesian magnetic field vector as its output. The sixth cell creates altitude and longitude profiles of the forces and inpact risk. Later sections contain these same profiles but with the simulation parameters changed.  
### Demonstration LEO
The first cell sets the simulation parameters to a very basic LEO satellite orbit. The second cell creates an animation that will demonstrate the physics of the electromagnetic forces on the tether. 
### Subsequent Sections
These sections set the simulation parameters to the orbits and tether dimensions of various different mission scenarios: 
DRAGRACER: Short tape-like tether in LEO
TSS-1R: Long wire-like tether in LEO
GTOSat: Short tape-like tether in GTO
After running the parameter reset cells, run animation and/or profile cells to see the simulation results for these new mission scenarios. 


Setup

In [ ]:
# Import GMAT

import sys
from os import path

# Create pathway to GMAT startup file
apistartup = "api_startup_file.txt"
GmatBinPath = "/Users/jode5596/Documents/GMAT R2025a/bin"
Startup = GmatBinPath + "/" + apistartup

# If the file exists, import the GMAT module and activate the startup file
if path.exists(Startup):

   sys.path.insert(1, GmatBinPath) # Update system path to give access to gmatpy module
   
   import gmatpy as gmat
   gmat.Setup(Startup)

# If the file doesn't exist in path, throw error text
else:
   print("Cannot find ", Startup)
   print()
   print("Please set up a GMAT startup file named ", apistartup, " in the ", 
      GmatBinPath, " folder.")
      

In [ ]:
# Import libraries / Create Satellite, Force Model, and Propagator
import numpy as np
from tqdm import tqdm  
from datetime import datetime, timedelta
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# Create satellite
sat = gmat.Construct("Spacecraft", "GTOSat")

# Orbit parameters
sat.SetField("DateFormat", "UTCGregorian")
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")  # Start time
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")  # Earth centered and non-rotating - The xy-plane of the system is aligned with the Earth's mean equatorial plane at the J2000 epoch. The X-axis points along the intersection of the Earth's mean equatorial plane and the mean ecliptic plane (at the J2000 epoch), in the direction of Aries
sat.SetField("DisplayStateType", "ModifiedKeplerian")
# Parameters for different mission scenarios to the right
sat.SetField("RadPer", 7078)      # DRAG(500km) = 6878   GTO1(185km) = 6378      GTO2(180) = 6558        TSS-1R(300) =  6678    Pavel = 6558
sat.SetField("RadApo", 7078.01)   # DRAG(500km) = 6878   GTO1(35787km) = 42165   GTO2(R=6.6Re) = 42096   TSS-1R(300) =  6678    Pavel(R=6.1Re) = 32212
sat.SetField("INC", 28.5)         # DRAG(97) = 97.37     GTO1 = 27               GTO1 = 27               TSS-1R = 28.5          Pavel = 25.6
sat.SetField("RAAN", 0)           # DRAG = 100           GTO1 = 0                GTO2 = 0                TSS-1R = 0? 
sat.SetField("AOP", 0)            # DRAG = 330           GTO1 = 180              GTO2 = 90               TSS-1R = 0? 
sat.SetField("TA", 0)             # All = 0
# Notes: Earth Mean Radius = 6371  Earth Max Radius = 6378

# Tether parameters (CSST=CubeSat Satellite Tether)
CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.28            # CCST = 0.075m     TSS-1R: 0.00254    DRAG: 0.154
L = 250               # CSST = 20m        TSS-1R: 19695    DRAG: 70
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat

# Satellite parameters
Mass1 = 12      # CubeSat Mass (kg)
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Mass2 = Mass1 + CSST_Mass # Tethered Mass
Area2 = Area1 + CSST_Area # Tethered Area
Cd1 = 2.2       # Drag coefficient of satellite

sat.SetField("DryMass", Mass2)
sat.SetField("Cd", Cd1)  # Drag coefficient
sat.SetField("Cr", 1.8)  # Radiation pressure coefficient (From Antoine and Pavels' code)
sat.SetField("DragArea", Area2)
sat.SetField("SRPArea", Area2)

# Create force model named "FM"
fm = gmat.Construct("ForceModel", "FM")

# Earth gravity (20x20 JGM-3)
earthgrav = gmat.Construct("GravityField")
earthgrav.SetField("BodyName","Earth")
earthgrav.SetField("Degree",20)
earthgrav.SetField("Order",20)
earthgrav.SetField("PotentialFile","JGM3.cof")

# Point mass gravity for Sun and Moon
moongrav = gmat.Construct("PointMassForce")
moongrav.SetField("BodyName","Luna")
sungrav = gmat.Construct("PointMassForce")
sungrav.SetField("BodyName","Sun")

# Drag using MSISE90
MS9drag = gmat.Construct("DragForce")
MS9drag.SetField("AtmosphereModel","MSISE90") 
atmos = gmat.Construct("MSISE90")  # Only GMAT atm model available
MS9drag.SetReference(atmos)


MS9drag.PredictedWeatherSource = 'ConstantFluxAndGeoMag'
MS9drag.CSSISpaceWeatherFile = 'SpaceWeather-All-v1.2.txt'
MS9drag.SchattenFile = 'SchattenPredict.txt'
MS9drag.SetField('F107',150)         # Solar Min: 70, Solar Max: 195, Average: 150 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression#:~:text=Solar%20cycle%20predictions%20are%20used,weather%20in%20the%20coming%20years.
MS9drag.SetField('F107A',150)
MS9drag.SetField('MagneticIndex',3)  # Kp index
MS9drag.SchattenErrorModel = 'Nominal'
MS9drag.SchattenTimingModel = 'NominalCycle'
MS9drag.DragModel = 'Spherical'

# Solar Radiation Pressure
SRP = gmat.SolarRadiationPressure("SRP")

# Add forces to model
fm.AddForce(earthgrav)
fm.AddForce(moongrav)
fm.AddForce(sungrav)
fm.AddForce(MS9drag)
fm.AddForce(SRP)

# Propagator

# Build the propagation container that connect the integrator, force model, and spacecraft together
pdprop = gmat.Construct("Propagator","PDProp")

# Create and assign a numerical integrator for use in the propagation
gator = gmat.Construct("RungeKutta89", "Gator")  # Options: PrinceDormand78 #RungeKutta89
pdprop.SetReference(gator)

# Set some of the fields for the integration
pdprop.SetField("InitialStepSize", 60.0)  
pdprop.SetField("Accuracy", 1.0e-12)    # Sets the error tolerance to 1×10⁻¹² (very high precision)
pdprop.SetField("MinStep", 0.0)         # Allows the integrator to take very small steps if needed for accuracy

# Assign the force model and satellite to the propagator
pdprop.SetReference(fm)
pdprop.AddPropObject(sat)

# Perform top level initialization
gmat.Initialize()

# Perform the integation subsysem initialization
pdprop.PrepareInternals()

# Refresh the integrator reference
gator = pdprop.GetPropagator()


Functions

In [ ]:
# Impact Risk
import numpy as np

def debris_density(sat_position_earthmj2000eq):
    """
    Calculate debris flux at satellite position using Chapman function.
    
    Parameters:
    sat_position_earthmj2000eq : array-like (x, y, z) in meters in EarthMJ2000Eq frame
    
    Returns:
    debris_density : float in particles/m^3
    """
    # Constants
    H1 = 300e3 #165e3  
    H2 = 200e3
    H3 = 300e3       
    N1 = 1e-15    #objects/m^3   # LEO Peak 1
    N2 = 5e-16    #objects/m^3   # LEO Peak 2
    N3 = 3.52e-18   #collisions/m^2/s   # GEO Peak       
       

    alt = np.linalg.norm(sat_position_earthmj2000eq) - 6378137


    h_peak_1 = 800e3       # LEO Peak 1
    h_peak_2 = 1500e3      # LEO Peak 2
    h_peak_3 = 35786e3     # GEO Peak

    
    # Calculate Chapman function for dayside (0° < SZA < 90°)
    

    z1 = (alt - h_peak_1) / H1
    z2 = (alt - h_peak_2) / H2
    z3 = (alt - h_peak_3) / H3


    # GEO Peak
    # https://conference.sdo.esoc.esa.int/proceedings/sdc9/paper/118/SDC9-paper118.pdf 
    # For >1cm we get 1e-5 collisions/30m^2/year
    # 1e-5 / 30 = 3.33e-7 collisions/m^2/year (flux)
    # flux (#/m^2/s) / v (m/s) = density (#/m^3)
    # 3.33e-7 collisions/m^2/year = 1.057e-14 collisions/m^2/s
    # 1.057e-14 / 3000m/s (GEO speed) = 3.52e-18 particles/m^3 = 3.5e-9 particles/km

    # LEO Peak
    # https://ntrs.nasa.gov/api/citations/20210011563/downloads/ORDEM_MASTER_ECSD_paper_Final_submitted%20v2.pdf 
    # 1e-6 particles/km^3 = 1e-15 particles/m^3
    # 1e-15 particles/m^3 * 7784m/s (800km orbital speed) = 7.784e-12 collisions/m^2/s = 2.45e-4 collisions/m^2/year

    # Find point of comparison between two sources
    # Use density w/ variable satellite speed instead of flux and compare

    debris_density1 = N1 * np.exp(1 - z1 - np.exp(-z1)) # Chapman function

    debris_density2 = N2 * np.exp(1 - z2 - np.exp(-z2)) # Chapman function

    debris_density3 = N3 * np.exp(1 - z3 - np.exp(-z3)) # Chapman function

    Minimum_flux = 1e-18  # Order of magnitude less than GEO


    debris_density = debris_density1 + debris_density2 + debris_density3 + Minimum_flux

    return debris_density


In [ ]:
# B-field function

import numpy as np                    # Arrays and such
from datetime import datetime         # Date time format
import matplotlib.pyplot as plt       # Plotting tools
import ppigrf                         # International Geomagnetic Reference Field model

# Position -> B-field function
def igrf_cartesian(current_date, r):
    """
    Calculate IGRF magnetic field in Cartesian coordinates at a given position and date.
    
    Parameters:
    current_date (datetime): Date for IGRF calculation
    r (array-like): 3D position vector in Cartesian coordinates [x, y, z] in km (Earth-centered)
    
    Returns:
    ndarray: Magnetic field vector in Cartesian coordinates [Bx, By, Bz] in nT
    """
    # Convert Cartesian to spherical coordinates
    x, y, z = r
    r_mag = np.sqrt(x**2 + y**2 + z**2)  # radial distance (km)
    
    # Calculate latitude and longitude (in degrees)
    theta = np.arccos(z / r_mag)  # colatitude in radians
    phi = np.arctan2(y, x)        # longitude in radians    actan2(y,x) is arctangent of y/x  arctan(y,x) only covers pi/2 to -pi/s
    
    # Convert to degrees (ppigrf expects degrees)
    theta_deg = np.degrees(theta)  # colatitude (0° at North pole, 180° at South pole)
    phi_deg = np.degrees(phi)      # longitude (0° to 360°)
    
    # Get IGRF components in spherical coordinates (nT)
    Br, Btheta, Bphi = ppigrf.igrf_gc(r_mag, theta_deg, phi_deg, current_date)
    
    # Convert spherical magnetic components to Cartesian coordinates  (Vector basis transformation)
    Bx = Br * np.sin(theta) * np.cos(phi) + Btheta * np.cos(theta) * np.cos(phi) - Bphi * np.sin(phi)
    By = Br * np.sin(theta) * np.sin(phi) + Btheta * np.cos(theta) * np.sin(phi) + Bphi * np.cos(phi)
    Bz = Br * np.cos(theta) - Btheta * np.sin(theta)
    
    return np.array([Bx, By, Bz])


In [ ]:
# Plasma density function
import PyIRI
import PyIRI.edp_update as ml
import numpy as np
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, ITRS, GCRS, CartesianRepresentation, get_sun
import astropy.units as u

def calculate_plasma_density(sat_position_earthmj2000eq, observation_time):
    """
    Calculate plasma density using PyIRI (International Reference Ionosphere).
    
    Parameters:
    sat_position_earthmj2000eq : array-like (x, y, z) in meters in EarthMJ2000Eq frame
    observation_time : datetime object
    
    Returns:
    plasma_density : float in particles/m^3
    solar_zenith : float in degrees
    """
    
    # Convert position to ITRS coordinates
    observation_time_astropy = Time(observation_time)
    
    cart_rep = CartesianRepresentation(
        x=sat_position_earthmj2000eq[0] * u.m,
        y=sat_position_earthmj2000eq[1] * u.m,
        z=sat_position_earthmj2000eq[2] * u.m
    )
    
    gcrs_coord = GCRS(cart_rep, obstime=observation_time_astropy)
    itrs_coord = gcrs_coord.transform_to(ITRS(obstime=observation_time_astropy))
    earth_location = EarthLocation.from_geocentric(itrs_coord.x, itrs_coord.y, itrs_coord.z)
    
    # Extract coordinates
    lat_deg = earth_location.lat.deg
    lon_deg = earth_location.lon.deg
    alt_km = earth_location.height.to(u.km).value
    
    # Get time components
    year = observation_time.year
    month = observation_time.month
    day = observation_time.day
    hour = observation_time.hour
    minute = observation_time.minute
    second = observation_time.second
    hour_dec = hour + minute/60.0 + second/3600.0
    
    # Calculate solar zenith angle
    sun_itrs = get_sun(observation_time_astropy).transform_to(ITRS(obstime=observation_time_astropy))
    
    sat_vec = np.array([itrs_coord.cartesian.x.value,
                        itrs_coord.cartesian.y.value,
                        itrs_coord.cartesian.z.value])
    sun_vec = np.array([sun_itrs.cartesian.x.value,
                        sun_itrs.cartesian.y.value,
                        sun_itrs.cartesian.z.value])
    
    dot_product = np.dot(sat_vec, sun_vec)
    norm_sat = np.linalg.norm(sat_vec)
    norm_sun = np.linalg.norm(sun_vec)
    cos_theta = dot_product / (norm_sat * norm_sun)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    solar_zenith = np.degrees(np.arccos(cos_theta))
    
    # Set up altitude array (need multiple altitudes for PyIRI to compute profile)
    # PyIRI requires an altitude array to calculate electron density profile
    # We'll create a single-point altitude array around our target altitude
    aalt = np.array([max(alt_km - 10, 100), alt_km, alt_km + 10])  # 3-point profile
    
    # Time array (single time point)
    ahr = np.array([hour_dec])
    
    # Location arrays (single point)
    alon = np.array([lon_deg])
    alat = np.array([lat_deg])
    
    # Solar flux (F10.7) - you can adjust these values
    f107 = 195.0  # Daily solar flux    Solar Min: 70, Solar Max: 195, Average: 150 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression#:~:text=Solar%20cycle%20predictions%20are%20used,weather%20in%20the%20coming%20years. 
    
    # IRI model options
    # ccir_or_ursi: 0 for CCIR, 1 for URSI
    ccir_or_ursi = 0
    
    try:
        # Get PyIRI coefficients directory path
        # The package should have a coeff_dir attribute
        coeff_dir = PyIRI.coeff_dir
        
        # Calculate electron density using PyIRI's daily density function
        # This returns: f2, f1, e_peak, es_peak, sun, mag, edp
        f2, f1, e_peak, es_peak, sun, mag, edp = ml.IRI_density_1day(
            year, month, day, ahr, alon, alat, aalt, f107, coeff_dir, ccir_or_ursi
        )
        
        # edp is the electron density profile [time, altitude, location]
        # Extract electron density at our target altitude
        # Find the closest altitude index
        alt_idx = np.argmin(np.abs(aalt - alt_km))
        plasma_density = edp[0, alt_idx, 0]  # [time_idx, alt_idx, loc_idx]
        
        # Ensure within reasonable bounds
        plasma_density = np.clip(plasma_density, 1e8, 1e13)
        
    except Exception as e:
        print(f"Warning: PyIRI calculation failed at ({lat_deg:.1f}°, {lon_deg:.1f}°, {alt_km:.1f} km)")
        print(f"Error: {e}")
        print("Using fallback Chapman model...")
        
        # Fallback to Chapman model
        H = 165e3
        N0 = 1e12
        N0_night = 1e11
        h_peak = 300e3
        height = alt_km * 1000
        
        if solar_zenith < 75:
            z = (height - h_peak) / H
            sec_chi = 1 / np.cos(np.radians(min(solar_zenith, 85)))
            plasma_density = N0 * np.exp(1 - z - sec_chi * np.exp(-z))
        else:
            z = (height - h_peak) / H
            plasma_density = N0_night * np.exp(1 - z - np.exp(-z))
        
        plasma_density = np.clip(plasma_density, 1e8, 1e13)
    
    return plasma_density, solar_zenith

In [ ]:
# Current/Length function

e = 1.60217663e-19                     # elementary charge (coulombs)
Me = 9.1093837e-31                     # electron mass (kg)
Mi = 1.67262192e-27                    # Ion (proton) mass (kg)

# Electron current per unit length      (Electron current dominates over ion current)   Source: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat 
def I_func(r,time,Vemf):
    n = calculate_plasma_density(r,time)[0]
    Ie = (2*w)*e*(n/np.pi)*np.sqrt(2*e*Vemf / Me)
    Iion = -(2*w)*e*(n/np.pi)*np.sqrt(2*e*(Vemf) / Mi)
    return Ie + Iion 


In [ ]:
# Atmospheric density function (NRLMSISE-00)

import numpy as np
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, ITRS, GCRS, CartesianRepresentation
import astropy.units as u
from nrlmsise00 import msise_model

def atmospheric_density_nrlmsise00(sat_position_earthmj2000eq, observation_time, f107=140.0, f107a=140.0, ap=1):
    """
    Calculate atmospheric density using NRLMSISE-00 model.
    
    Parameters:
    - sat_position_earthmj2000eq : array-like (x, y, z) in meters (EarthMJ2000Eq frame)
    - observation_time : datetime object
    - f107 : Daily F10.7 solar flux (default: 140)
    - f107a : 81-day avg F10.7 flux (default: 140)
    - ap : Geomagnetic Ap index (default: 6)
    
    Returns:
    - Atmospheric density (kg/m³)
    - Solar Zenith Angle (degrees)
    """
    # Convert satellite position to geodetic coordinates (lat, lon, alt)
    observation_time_astropy = Time(observation_time)
    cart_rep = CartesianRepresentation(
        x=sat_position_earthmj2000eq[0] * u.m,
        y=sat_position_earthmj2000eq[1] * u.m,
        z=sat_position_earthmj2000eq[2] * u.m
    )
    gcrs_coord = GCRS(cart_rep, obstime=observation_time_astropy)
    itrs_coord = gcrs_coord.transform_to(ITRS(obstime=observation_time_astropy))
    earth_location = EarthLocation.from_geocentric(itrs_coord.x, itrs_coord.y, itrs_coord.z)
    
    # Extract latitude, longitude, altitude (km)
    lat = earth_location.lat.deg
    lon = earth_location.lon.deg
    alt_km = earth_location.height.to(u.km).value  # NRLMSISE-00 expects km

    # Get solar zenith angle (SZA)
    from astropy.coordinates import get_sun
    sun_itrs = get_sun(observation_time_astropy).transform_to(ITRS(obstime=observation_time_astropy))
    sat_vec = np.array([itrs_coord.cartesian.x.value, itrs_coord.cartesian.y.value, itrs_coord.cartesian.z.value])
    sun_vec = np.array([sun_itrs.cartesian.x.value, sun_itrs.cartesian.y.value, sun_itrs.cartesian.z.value])
    cos_theta = np.dot(sat_vec, sun_vec) / (np.linalg.norm(sat_vec) * np.linalg.norm(sun_vec))
    solar_zenith = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))

    # Compute atmospheric density using NRLMSISE-00
    output = msise_model(
        time=observation_time,
        alt=alt_km,
        lat=lat,
        lon=lon,
        f107=f107,
        f107a=f107a,
        ap=ap
    )
    
    # The output list of float where the sixth element is total mass density (g/cm³)
    total_density_gcm3 = output[0][5]  # total mass density [g cm^-3]
    density_kgm3 = total_density_gcm3 * 1e3  # Convert g/cm³ → kg/m³

    return density_kgm3, solar_zenith
    #return output

# Example usage
sat_position = [7000e3, 0, 0]  # 7000 km in x-direction (approx. 622 km altitude)
obs_time = datetime(2024, 1, 1, 12, 0, 0)
density, sza = atmospheric_density_nrlmsise00(sat_position, obs_time)
print(f"Density: {density} kg/m³, SZA: {sza:.1f}°")


In [ ]:
# Combined Altitude and Longitude Profiles Analysis (5° Off-Vertical Tether Orientation)
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
TETHER_ANGLE = np.radians(5)  # 5 degrees off vertical (radians)

# Configuration - Flexible altitude values for longitude profiles
LONGITUDE_ALTITUDES = [300, 700]  # km - can be modified

# Single observation time (converted to Astropy Time)
obs_time = Time(datetime(2025, 3, 20, 12, 0, 0))  # Fixed universal time (Equinox)

# Find subsolar and anti-subsolar points at this time
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg  # Subsolar longitude
anti_subsolar_lon = (subsolar_lon + 180) % 360  # Opposite side

# Altitude range (100-1000 km) for altitude profiles
altitudes_km = np.linspace(100, 1000, 50)
altitudes_m = altitudes_km * 1000 + R_EARTH

# Longitude range (0-360° in 5° steps) for longitude profiles
longitudes = np.linspace(0, 360, 73)  # 5° resolution

# Initialize arrays for storage
results_altitude = {
    'noon': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    },
    'midnight': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }
}

results_longitude = {}
for alt in LONGITUDE_ALTITUDES:
    results_longitude[f"{alt}km"] = {
        'longitude': [], 'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }

def get_position(alt_m, lon_deg):
    """Get position vector for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    return np.array([x, y, z]), np.array([x/1000, y/1000, z/1000])  # Return both m and km versions

def get_position_and_tether_dir(alt_m, lon_deg):
    """Get position vector and tether direction for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    
    # Position vectors (m and km)
    r_m = np.array([x, y, z])
    r_km = r_m / 1000
    
    # Calculate directions
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Velocity vector (tangential to orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)
    v_vec = np.array([-v_mag*np.sin(np.radians(lon_deg)), 
                      v_mag*np.cos(np.radians(lon_deg)), 
                      0])
    
    # Calculate along-track direction
    v_perp = v_vec - np.dot(v_vec, radial_dir) * radial_dir
    along_track_dir = v_perp / np.linalg.norm(v_perp) if np.linalg.norm(v_perp) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir = np.cos(TETHER_ANGLE) * radial_dir + np.sin(TETHER_ANGLE) * along_track_dir
    
    return r_m, r_km, tether_dir, v_vec

# Calculate altitude profiles
print("Calculating altitude profiles...")
for alt_m, alt_km in zip(altitudes_m, altitudes_km):
    # Subsolar (noon) position
    r_m_noon, r_noon = get_position(alt_m, subsolar_lon)
    # Anti-subsolar (midnight) position
    r_m_midnight, r_midnight = get_position(alt_m, anti_subsolar_lon)
    
    # Velocity vectors (circular orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)  # Magnitude
    v_vec_noon = np.array([0, v_mag, 0])  # Simplified - would need proper transformation
    v_vec_midnight = np.array([0, -v_mag, 0])  # Opposite direction
    
    # Calculate radial and along-track directions
    radial_dir_noon = r_m_noon / np.linalg.norm(r_m_noon)
    v_perp_noon = v_vec_noon - np.dot(v_vec_noon, radial_dir_noon) * radial_dir_noon
    along_track_dir_noon = v_perp_noon / np.linalg.norm(v_perp_noon) if np.linalg.norm(v_perp_noon) > 0 else np.array([1,0,0])
    
    radial_dir_midnight = r_m_midnight / np.linalg.norm(r_m_midnight)
    v_perp_midnight = v_vec_midnight - np.dot(v_vec_midnight, radial_dir_midnight) * radial_dir_midnight
    along_track_dir_midnight = v_perp_midnight / np.linalg.norm(v_perp_midnight) if np.linalg.norm(v_perp_midnight) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir_noon = (np.cos(TETHER_ANGLE) * radial_dir_noon + 
                      np.sin(TETHER_ANGLE) * along_track_dir_noon)
    tether_dir_midnight = (np.cos(TETHER_ANGLE) * radial_dir_midnight + 
                         np.sin(TETHER_ANGLE) * along_track_dir_midnight)
    
    for condition, r_m, r, v_vec, tether_dir in [
        ('noon', r_m_noon, r_noon, v_vec_noon, tether_dir_noon),
        ('midnight', r_m_midnight, r_midnight, v_vec_midnight, tether_dir_midnight)
    ]:
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_altitude[condition]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_altitude[condition]['plasma_density'].append(n)
        results_altitude[condition]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results_altitude[condition]['atmos_density'].append(rho)
        
        # Velocity
        results_altitude[condition]['velocity'].append(v_mag)
        
        # Vemf (full tether length)
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_altitude[condition]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)        # F prop L
        results_altitude[condition]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_altitude[condition]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_altitude[condition]['lorentz_force'].append(F_lorentz)

# Calculate longitude profiles
print("Calculating longitude profiles...")
for alt_km in LONGITUDE_ALTITUDES:
    alt_m = alt_km * 1000 + R_EARTH
    key = f"{alt_km}km"
    
    for lon in longitudes:
        r_m, r_km, tether_dir, v_vec = get_position_and_tether_dir(alt_m, lon)
        v_mag = np.linalg.norm(v_vec)
        
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r_km) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_longitude[key]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_longitude[key]['plasma_density'].append(n)
        results_longitude[key]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results_longitude[key]['atmos_density'].append(rho)
        
        # Velocity
        results_longitude[key]['velocity'].append(v_mag)
        
        # Vemf
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_longitude[key]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)
        results_longitude[key]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_longitude[key]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_longitude[key]['lorentz_force'].append(F_lorentz)
        
        # Longitude
        results_longitude[key]['longitude'].append(lon)

# NEW FUNCTIONALITY: Print values at specified altitude
def print_values_at_altitude(altitude_km):
    """Print all parameter values at a specified altitude"""
    # Find the closest altitude in our data
    idx = np.argmin(np.abs(altitudes_km - altitude_km))
    actual_altitude = altitudes_km[idx]
    
    print(f"\n{'='*80}")
    print(f"PARAMETER VALUES AT {actual_altitude:.1f} km ALTITUDE")
    print(f"{'='*80}")
    
    print(f"{'Parameter':<25} {'Noon':<15} {'Midnight':<15} {'Units':<10}")
    print(f"{'-'*80}")
    
    # Magnetic Field
    print(f"{'Magnetic Field':<25} {results_altitude['noon']['B_field'][idx]:<15.2e} {results_altitude['midnight']['B_field'][idx]:<15.2e} {'T':<10}")
    
    # Plasma Density
    print(f"{'Plasma Density':<25} {results_altitude['noon']['plasma_density'][idx]:<15.2e} {results_altitude['midnight']['plasma_density'][idx]:<15.2e} {'m⁻³':<10}")
    
    # Atmospheric Density
    print(f"{'Atmospheric Density':<25} {results_altitude['noon']['atmos_density'][idx]:<15.2e} {results_altitude['midnight']['atmos_density'][idx]:<15.2e} {'kg/m³':<10}")
    
    # Velocity
    print(f"{'Orbital Velocity':<25} {results_altitude['noon']['velocity'][idx]:<15.2f} {results_altitude['midnight']['velocity'][idx]:<15.2f} {'m/s':<10}")
    
    # Induced EMF
    print(f"{'Induced EMF (Vemf)':<25} {results_altitude['noon']['Vemf'][idx]:<15.2f} {results_altitude['midnight']['Vemf'][idx]:<15.2f} {'V':<10}")
    
    # Current
    print(f"{'Tether Current':<25} {results_altitude['noon']['current'][idx]:<15.2e} {results_altitude['midnight']['current'][idx]:<15.2e} {'A':<10}")
    
    # Drag Force
    print(f"{'Drag Force':<25} {results_altitude['noon']['drag_force'][idx]:<15.2e} {results_altitude['midnight']['drag_force'][idx]:<15.2e} {'N':<10}")
    
    # Lorentz Force
    print(f"{'Lorentz Force':<25} {results_altitude['noon']['lorentz_force'][idx]:<15.2e} {results_altitude['midnight']['lorentz_force'][idx]:<15.2e} {'N':<10}")
    
    # SZA
    print(f"{'Solar Zenith Angle':<25} {results_altitude['noon']['sza'][idx]:<15.1f} {results_altitude['midnight']['sza'][idx]:<15.1f} {'°':<10}")
    
    # Force Ratio
    ratio_noon = results_altitude['noon']['lorentz_force'][idx] / results_altitude['noon']['drag_force'][idx]
    ratio_midnight = results_altitude['midnight']['lorentz_force'][idx] / results_altitude['midnight']['drag_force'][idx]
    print(f"{'Lorentz/Drag Ratio':<25} {ratio_noon:<15.2f} {ratio_midnight:<15.2f} {'':<10}")
    
    # Net Force
    net_noon = results_altitude['noon']['lorentz_force'][idx] - results_altitude['noon']['drag_force'][idx]
    net_midnight = results_altitude['midnight']['lorentz_force'][idx] - results_altitude['midnight']['drag_force'][idx]
    print(f"{'Net Force (L-D)':<25} {net_noon:<15.2e} {net_midnight:<15.2e} {'N':<10}")
    
    print(f"{'='*80}")

# Print values at 700 km
print_values_at_altitude(700)

# Create comprehensive figure with larger fonts
plt.figure(figsize=(25, 30))
plt.suptitle(f'Electrodynamic Tether Analysis (5° Off-Vertical Orientation)\n'
             f'Observation Time: {obs_time.datetime.strftime("%Y-%m-%d %H:%M UTC")}, '
             f'Subsolar Longitude: {subsolar_lon:.1f}°',
             fontsize=20, fontweight='bold', y=1.02)


# Set global font sizes
label_fontsize = 19
title_fontsize = 22
legend_fontsize = 16
tick_fontsize = 16

# Vertical lines for subsolar/anti-subsolar points
subsol_line = {'ls': '--', 'color': 'gold', 'alpha': 0.7, 'label': 'Subsolar'}
antisol_line = {'ls': '--', 'color': 'navy', 'alpha': 0.7, 'label': 'Anti-subsolar'}

# Row 1-2: Altitude Profiles (8 subplots)
# 1. Magnetic Field vs Altitude
plt.subplot(4, 4, 1)
plt.plot(results_altitude['midnight']['B_field'], altitudes_km, label='Midnight (Anti-subsolar)')
plt.plot(results_altitude['noon']['B_field'], altitudes_km, label='Noon (Subsolar)')
plt.xlabel('Magnetic Field Strength (T)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(a) Magnetic Field vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 2. Plasma Density vs Altitude
plt.subplot(4, 4, 2)
plt.semilogx(results_altitude['midnight']['plasma_density'], altitudes_km, 
             label=f'Midnight (SZA={results_altitude["midnight"]["sza"][0]:.1f}°)')
plt.semilogx(results_altitude['noon']['plasma_density'], altitudes_km, 
             label=f'Noon (SZA={results_altitude["noon"]["sza"][0]:.1f}°)')
plt.xlabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(b) Plasma Density vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 3. Atmospheric Density vs Altitude
plt.subplot(4, 4, 3)
plt.semilogx(results_altitude['midnight']['atmos_density'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['atmos_density'], altitudes_km, label='Noon')
plt.xlabel('Atmospheric Density (kg/m$^3$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(c) Atmospheric Density vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 4. Orbital Velocity vs Altitude
plt.subplot(4, 4, 4)
plt.plot(results_altitude['midnight']['velocity'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['velocity'], altitudes_km, label='Noon')
plt.xlabel('Orbital Velocity (m/s)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(d) Circular Orbit Velocity vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 5. Induced EMF vs Altitude
plt.subplot(4, 4, 5)
plt.plot(results_altitude['midnight']['Vemf'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['Vemf'], altitudes_km, label='Noon')
plt.xlabel('Induced EMF (V)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(e) Tether EMF vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 6. Current vs Altitude
plt.subplot(4, 4, 6)
plt.semilogx(results_altitude['midnight']['current'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['current'], altitudes_km, label='Noon')
plt.xlabel('Current (A)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(f) Tether Current vs Altitude\n(Dominated by Plasma Density)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 7. Drag Force vs Altitude
plt.subplot(4, 4, 7)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, label='Noon')
plt.xlabel('Drag Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(g) Atmospheric Drag vs Altitude', fontsize=title_fontsize)
#plt.xlim(1e-5,1e4)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 8. Lorentz Force vs Altitude
plt.subplot(4, 4, 8)
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, label='Noon')
plt.xlabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(h) Electrodynamic Tether Force vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
#plt.xlim(1e-5,1e4)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 3: Longitude Profiles (4 key parameters)
# 9. Magnetic Field vs Longitude
plt.subplot(4, 4, 9)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.plot(results_longitude[key]['longitude'], results_longitude[key]['B_field'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Magnetic Field (T)', fontsize=label_fontsize)
plt.title('(i) B-field vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 10. Plasma Density vs Longitude
plt.subplot(4, 4, 10)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['plasma_density'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.title('(j) Plasma Density vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 11. Drag Force vs Longitude
plt.subplot(4, 4, 11)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['drag_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Drag Force (N)', fontsize=label_fontsize)
plt.title('(k) Atmospheric Drag vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 12. Lorentz Force vs Longitude
plt.subplot(4, 4, 12)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['lorentz_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.title('(l) Electrodynamic Tether Force vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 4: Force Comparisons and Ratios
# 13. Altitude: Drag vs Lorentz Force Comparison (Noon)
plt.subplot(4, 4, 13)
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, 'r-', label='Drag (Noon)')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, 'r--', label='Lorentz (Noon)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(m) Drag vs Lorentz Force (Noon)', fontsize=title_fontsize)
plt.xlim(1e-5,1e4)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

#blue: #0072B2     orange: #E69F00


# 14. Altitude: Drag vs Lorentz Force Comparison (Midnight)
plt.subplot(4, 4, 14)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, 'b-', label='Drag (Midnight)')
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, 'b--', label='Lorentz (Midnight)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(n) Drag vs Lorentz Force (Midnight)', fontsize=title_fontsize)
plt.xlim(1e-5,1e4)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 15. Altitude: Force Ratio
plt.subplot(4, 4, 15)
ratio_noon = np.array(results_altitude['noon']['lorentz_force']) / np.array(results_altitude['noon']['drag_force'])
ratio_midnight = np.array(results_altitude['midnight']['lorentz_force']) / np.array(results_altitude['midnight']['drag_force'])
plt.semilogx(ratio_midnight, altitudes_km, label='Midnight (Lorentz/Drag)') # 'm-', 
plt.semilogx(ratio_noon, altitudes_km, label='Noon (Lorentz/Drag)') #'g-',
plt.axvline(x=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(o) Lorentz to Drag Force Ratio', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 16. Longitude: Force Ratio
plt.subplot(4, 4, 16)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    ratio = np.array(results_longitude[key]['lorentz_force']) / np.array(results_longitude[key]['drag_force'])
    plt.semilogy(results_longitude[key]['longitude'], ratio, label=f'{alt}km (Lorentz/Drag)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.title('(p) Lorentz to Drag Force Ratio vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

Demonstration LEO

In [ ]:
# Reset to 28 deg LEO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "20 Nov 2020 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6878)   
sat.SetField("RadApo", 6878.005)
sat.SetField("INC", 0)  
sat.SetField("RAAN", 100)
sat.SetField("AOP", 330)
sat.SetField("TA", 0) 


CSST_Mass = 0.083      # CSST = 0.083kg    TSS-1R: ?
w = 0.154            # CCST = 0.075m     TSS-1R: 0.00254      DRAGRACER = 0.154 cm
L = 70               # CSST = 20m        TSS-1R: 19695        DRAGRACER = 70m
CSST_Area = (2 / np.pi) * w * L  




# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0005 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 0  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

#blue: #0072B2   orange: #E69F00 


# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

DRAGRACER

In [ ]:
# Reset to DRAGRACER LEO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "20 Nov 2020 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6878)   
sat.SetField("RadApo", 6878.005)
sat.SetField("INC", 97.37)  
sat.SetField("RAAN", 100)
sat.SetField("AOP", 330)
sat.SetField("TA", 0) 


CSST_Mass = 0.083      # CSST = 0.083kg    TSS-1R: ?
w = 0.154            # CCST = 0.075m     TSS-1R: 0.00254      DRAGRACER = 0.154 cm
L = 70               # CSST = 20m        TSS-1R: 19695        DRAGRACER = 70m
CSST_Area = (2 / np.pi) * w * L  


# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Combined Altitude and Longitude Profiles Analysis (5° Off-Vertical Tether Orientation)
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
TETHER_ANGLE = np.radians(5)  # 5 degrees off vertical (radians)

# Configuration - Flexible altitude values for longitude profiles
LONGITUDE_ALTITUDES = [300, 700]  # km - can be modified

# Single observation time (converted to Astropy Time)
obs_time = Time(datetime(2025, 3, 20, 12, 0, 0))  # Fixed universal time (Equinox)

# Find subsolar and anti-subsolar points at this time
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg  # Subsolar longitude
anti_subsolar_lon = (subsolar_lon + 180) % 360  # Opposite side

# Altitude range (100-1000 km) for altitude profiles
altitudes_km = np.linspace(100, 1000, 50)
altitudes_m = altitudes_km * 1000 + R_EARTH

# Longitude range (0-360° in 5° steps) for longitude profiles
longitudes = np.linspace(0, 360, 73)  # 5° resolution

# Initialize arrays for storage
results_altitude = {
    'noon': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    },
    'midnight': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }
}

results_longitude = {}
for alt in LONGITUDE_ALTITUDES:
    results_longitude[f"{alt}km"] = {
        'longitude': [], 'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }

def get_position(alt_m, lon_deg):
    """Get position vector for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    return np.array([x, y, z]), np.array([x/1000, y/1000, z/1000])  # Return both m and km versions

def get_position_and_tether_dir(alt_m, lon_deg):
    """Get position vector and tether direction for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    
    # Position vectors (m and km)
    r_m = np.array([x, y, z])
    r_km = r_m / 1000
    
    # Calculate directions
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Velocity vector (tangential to orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)
    v_vec = np.array([-v_mag*np.sin(np.radians(lon_deg)), 
                      v_mag*np.cos(np.radians(lon_deg)), 
                      0])
    
    # Calculate along-track direction
    v_perp = v_vec - np.dot(v_vec, radial_dir) * radial_dir
    along_track_dir = v_perp / np.linalg.norm(v_perp) if np.linalg.norm(v_perp) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir = np.cos(TETHER_ANGLE) * radial_dir + np.sin(TETHER_ANGLE) * along_track_dir
    
    return r_m, r_km, tether_dir, v_vec

# Calculate altitude profiles
print("Calculating altitude profiles...")
for alt_m, alt_km in zip(altitudes_m, altitudes_km):
    # Subsolar (noon) position
    r_m_noon, r_noon = get_position(alt_m, subsolar_lon)
    # Anti-subsolar (midnight) position
    r_m_midnight, r_midnight = get_position(alt_m, anti_subsolar_lon)
    
    # Velocity vectors (circular orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)  # Magnitude
    v_vec_noon = np.array([0, v_mag, 0])  # Simplified - would need proper transformation
    v_vec_midnight = np.array([0, -v_mag, 0])  # Opposite direction
    
    # Calculate radial and along-track directions
    radial_dir_noon = r_m_noon / np.linalg.norm(r_m_noon)
    v_perp_noon = v_vec_noon - np.dot(v_vec_noon, radial_dir_noon) * radial_dir_noon
    along_track_dir_noon = v_perp_noon / np.linalg.norm(v_perp_noon) if np.linalg.norm(v_perp_noon) > 0 else np.array([1,0,0])
    
    radial_dir_midnight = r_m_midnight / np.linalg.norm(r_m_midnight)
    v_perp_midnight = v_vec_midnight - np.dot(v_vec_midnight, radial_dir_midnight) * radial_dir_midnight
    along_track_dir_midnight = v_perp_midnight / np.linalg.norm(v_perp_midnight) if np.linalg.norm(v_perp_midnight) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir_noon = (np.cos(TETHER_ANGLE) * radial_dir_noon + 
                      np.sin(TETHER_ANGLE) * along_track_dir_noon)
    tether_dir_midnight = (np.cos(TETHER_ANGLE) * radial_dir_midnight + 
                         np.sin(TETHER_ANGLE) * along_track_dir_midnight)
    
    for condition, r_m, r, v_vec, tether_dir in [
        ('noon', r_m_noon, r_noon, v_vec_noon, tether_dir_noon),
        ('midnight', r_m_midnight, r_midnight, v_vec_midnight, tether_dir_midnight)
    ]:
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_altitude[condition]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_altitude[condition]['plasma_density'].append(n)
        results_altitude[condition]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime, f107=75.0, f107a=75.0)
        results_altitude[condition]['atmos_density'].append(rho)
        
        # Velocity
        results_altitude[condition]['velocity'].append(v_mag)
        
        # Vemf (full tether length)
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_altitude[condition]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)        # F prop L
        results_altitude[condition]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_altitude[condition]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_altitude[condition]['lorentz_force'].append(F_lorentz)

# Calculate longitude profiles
print("Calculating longitude profiles...")
for alt_km in LONGITUDE_ALTITUDES:
    alt_m = alt_km * 1000 + R_EARTH
    key = f"{alt_km}km"
    
    for lon in longitudes:
        r_m, r_km, tether_dir, v_vec = get_position_and_tether_dir(alt_m, lon)
        v_mag = np.linalg.norm(v_vec)
        
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r_km) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_longitude[key]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_longitude[key]['plasma_density'].append(n)
        results_longitude[key]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results_longitude[key]['atmos_density'].append(rho)
        
        # Velocity
        results_longitude[key]['velocity'].append(v_mag)
        
        # Vemf
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_longitude[key]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)
        results_longitude[key]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_longitude[key]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_longitude[key]['lorentz_force'].append(F_lorentz)
        
        # Longitude
        results_longitude[key]['longitude'].append(lon)

# NEW FUNCTIONALITY: Print values at specified altitude
def print_values_at_altitude(altitude_km):
    """Print all parameter values at a specified altitude"""
    # Find the closest altitude in our data
    idx = np.argmin(np.abs(altitudes_km - altitude_km))
    actual_altitude = altitudes_km[idx]
    
    print(f"\n{'='*80}")
    print(f"PARAMETER VALUES AT {actual_altitude:.1f} km ALTITUDE")
    print(f"{'='*80}")
    
    print(f"{'Parameter':<25} {'Noon':<15} {'Midnight':<15} {'Units':<10}")
    print(f"{'-'*80}")
    
    # Magnetic Field
    print(f"{'Magnetic Field':<25} {results_altitude['noon']['B_field'][idx]:<15.2e} {results_altitude['midnight']['B_field'][idx]:<15.2e} {'T':<10}")
    
    # Plasma Density
    print(f"{'Plasma Density':<25} {results_altitude['noon']['plasma_density'][idx]:<15.2e} {results_altitude['midnight']['plasma_density'][idx]:<15.2e} {'m⁻³':<10}")
    
    # Atmospheric Density
    print(f"{'Atmospheric Density':<25} {results_altitude['noon']['atmos_density'][idx]:<15.2e} {results_altitude['midnight']['atmos_density'][idx]:<15.2e} {'kg/m³':<10}")
    
    # Velocity
    print(f"{'Orbital Velocity':<25} {results_altitude['noon']['velocity'][idx]:<15.2f} {results_altitude['midnight']['velocity'][idx]:<15.2f} {'m/s':<10}")
    
    # Induced EMF
    print(f"{'Induced EMF (Vemf)':<25} {results_altitude['noon']['Vemf'][idx]:<15.2f} {results_altitude['midnight']['Vemf'][idx]:<15.2f} {'V':<10}")
    
    # Current
    print(f"{'Tether Current':<25} {results_altitude['noon']['current'][idx]:<15.2e} {results_altitude['midnight']['current'][idx]:<15.2e} {'A':<10}")
    
    # Drag Force
    print(f"{'Drag Force':<25} {results_altitude['noon']['drag_force'][idx]:<15.2e} {results_altitude['midnight']['drag_force'][idx]:<15.2e} {'N':<10}")
    
    # Lorentz Force
    print(f"{'Lorentz Force':<25} {results_altitude['noon']['lorentz_force'][idx]:<15.2e} {results_altitude['midnight']['lorentz_force'][idx]:<15.2e} {'N':<10}")
    
    # SZA
    print(f"{'Solar Zenith Angle':<25} {results_altitude['noon']['sza'][idx]:<15.1f} {results_altitude['midnight']['sza'][idx]:<15.1f} {'°':<10}")
    
    # Force Ratio
    ratio_noon = results_altitude['noon']['lorentz_force'][idx] / results_altitude['noon']['drag_force'][idx]
    ratio_midnight = results_altitude['midnight']['lorentz_force'][idx] / results_altitude['midnight']['drag_force'][idx]
    print(f"{'Lorentz/Drag Ratio':<25} {ratio_noon:<15.2f} {ratio_midnight:<15.2f} {'':<10}")
    
    # Net Force
    net_noon = results_altitude['noon']['lorentz_force'][idx] - results_altitude['noon']['drag_force'][idx]
    net_midnight = results_altitude['midnight']['lorentz_force'][idx] - results_altitude['midnight']['drag_force'][idx]
    print(f"{'Net Force (L-D)':<25} {net_noon:<15.2e} {net_midnight:<15.2e} {'N':<10}")
    
    print(f"{'='*80}")

# Print values at 500 km
print_values_at_altitude(500)

# Create comprehensive figure with larger fonts
plt.figure(figsize=(25, 30))
plt.suptitle(f'Electrodynamic Tether Analysis (5° Off-Vertical Orientation)\n'
             f'Observation Time: {obs_time.datetime.strftime("%Y-%m-%d %H:%M UTC")}, '
             f'Subsolar Longitude: {subsolar_lon:.1f}°',
             fontsize=20, fontweight='bold', y=1.02)


# Set global font sizes
label_fontsize = 19
title_fontsize = 22
legend_fontsize = 16
tick_fontsize = 16

# Vertical lines for subsolar/anti-subsolar points
subsol_line = {'ls': '--', 'color': 'gold', 'alpha': 0.7, 'label': 'Subsolar'}
antisol_line = {'ls': '--', 'color': 'navy', 'alpha': 0.7, 'label': 'Anti-subsolar'}

# Row 1-2: Altitude Profiles (8 subplots)
# 1. Magnetic Field vs Altitude
plt.subplot(4, 4, 1)
plt.plot(results_altitude['midnight']['B_field'], altitudes_km, label='Midnight (Anti-subsolar)')
plt.plot(results_altitude['noon']['B_field'], altitudes_km, label='Noon (Subsolar)')
plt.xlabel('Magnetic Field Strength (T)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(a) Magnetic Field vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 2. Plasma Density vs Altitude
plt.subplot(4, 4, 2)
plt.semilogx(results_altitude['midnight']['plasma_density'], altitudes_km, 
             label=f'Midnight (SZA={results_altitude["midnight"]["sza"][0]:.1f}°)')
plt.semilogx(results_altitude['noon']['plasma_density'], altitudes_km, 
             label=f'Noon (SZA={results_altitude["noon"]["sza"][0]:.1f}°)')
plt.xlabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(b) Plasma Density vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 3. Atmospheric Density vs Altitude
plt.subplot(4, 4, 3)
plt.semilogx(results_altitude['midnight']['atmos_density'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['atmos_density'], altitudes_km, label='Noon')
plt.xlabel('Atmospheric Density (kg/m$^3$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(c) Atmospheric Density vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 4. Orbital Velocity vs Altitude
plt.subplot(4, 4, 4)
plt.plot(results_altitude['midnight']['velocity'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['velocity'], altitudes_km, label='Noon')
plt.xlabel('Orbital Velocity (m/s)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(d) Circular Orbit Velocity vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 5. Induced EMF vs Altitude
plt.subplot(4, 4, 5)
plt.plot(results_altitude['midnight']['Vemf'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['Vemf'], altitudes_km, label='Noon')
plt.xlabel('Induced EMF (V)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(e) Tether EMF vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 6. Current vs Altitude
plt.subplot(4, 4, 6)
plt.semilogx(results_altitude['midnight']['current'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['current'], altitudes_km, label='Noon')
plt.xlabel('Current (A)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(f) Tether Current vs Altitude\n(Dominated by Plasma Density)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 7. Drag Force vs Altitude
plt.subplot(4, 4, 7)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, label='Noon')
plt.xlabel('Drag Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(g) Atmospheric Drag vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 8. Lorentz Force vs Altitude
plt.subplot(4, 4, 8)
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, label='Noon')
plt.xlabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(h) Electrodynamic Tether Force vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 3: Longitude Profiles (4 key parameters)
# 9. Magnetic Field vs Longitude
plt.subplot(4, 4, 9)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.plot(results_longitude[key]['longitude'], results_longitude[key]['B_field'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Magnetic Field (T)', fontsize=label_fontsize)
plt.title('(i) B-field vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 10. Plasma Density vs Longitude
plt.subplot(4, 4, 10)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['plasma_density'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.title('(j) Plasma Density vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 11. Drag Force vs Longitude
plt.subplot(4, 4, 11)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['drag_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Drag Force (N)', fontsize=label_fontsize)
plt.title('(k) Atmospheric Drag vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 12. Lorentz Force vs Longitude
plt.subplot(4, 4, 12)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['lorentz_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.title('(l) Electrodynamic Tether Force vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 4: Force Comparisons and Ratios
# 13. Altitude: Drag vs Lorentz Force Comparison (Noon)
plt.subplot(4, 4, 13)
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, 'r-', label='Drag (Noon)')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, 'r--', label='Lorentz (Noon)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(m) Drag vs Lorentz Force (Noon)', fontsize=title_fontsize)
plt.grid(True)
plt.xlim(1e-7,1e3)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 14. Altitude: Drag vs Lorentz Force Comparison (Midnight)
plt.subplot(4, 4, 14)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, 'b-', label='Drag (Midnight)')
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, 'b--', label='Lorentz (Midnight)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(n) Drag vs Lorentz Force (Midnight)', fontsize=title_fontsize)
plt.grid(True)
plt.xlim(1e-7,1e3)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 15. Altitude: Force Ratio
plt.subplot(4, 4, 15)
ratio_noon = np.array(results_altitude['noon']['lorentz_force']) / np.array(results_altitude['noon']['drag_force'])
ratio_midnight = np.array(results_altitude['midnight']['lorentz_force']) / np.array(results_altitude['midnight']['drag_force'])
plt.semilogx(ratio_midnight, altitudes_km, label='Midnight (Lorentz/Drag)') # 'm-', 
plt.semilogx(ratio_noon, altitudes_km, label='Noon (Lorentz/Drag)') #'g-',
plt.axvline(x=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(o) Lorentz to Drag Force Ratio', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 16. Longitude: Force Ratio
plt.subplot(4, 4, 16)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    ratio = np.array(results_longitude[key]['lorentz_force']) / np.array(results_longitude[key]['drag_force'])
    plt.semilogy(results_longitude[key]['longitude'], ratio, label=f'{alt}km (Lorentz/Drag)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.title('(p) Lorentz to Drag Force Ratio vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

In [ ]:
# Altitude Profiles at Noon and Midnight (Multiple Tether Orientations)
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
TETHER_ANGLES = np.radians([0, 5, 45, 85, 90])  # Multiple tether angles (radians)

# Single observation time (converted to Astropy Time)
obs_time = Time(datetime(2025, 3, 20, 12, 0, 0))  # Fixed universal time (Equinox)

# Find subsolar and anti-subsolar points at this time
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg  # Subsolar longitude
anti_subsolar_lon = (subsolar_lon + 180) % 360  # Opposite side

# Altitude range (100-1000 km)
altitudes_km = np.linspace(100, 1000, 50)
altitudes_m = altitudes_km * 1000 + R_EARTH

# Initialize results dictionary for all angles
results = {}
for angle_deg in [0, 5, 45, 85, 90]:
    results[angle_deg] = {
        'noon': {
            'B_field': [], 'plasma_density': [], 'atmos_density': [],
            'velocity': [], 'Vemf': [], 'current': [], 
            'drag_force': [], 'lorentz_force': [], 'sza': []
        },
        'midnight': {
            'B_field': [], 'plasma_density': [], 'atmos_density': [],
            'velocity': [], 'Vemf': [], 'current': [], 
            'drag_force': [], 'lorentz_force': [], 'sza': []
        }
    }

def get_position(alt_m, lon_deg):
    """Get position vector for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    return np.array([x, y, z]), np.array([x/1000, y/1000, z/1000])  # Return both m and km versions

# Calculate all parameters for all angles and both conditions
for alt_m, alt_km in zip(altitudes_m, altitudes_km):
    # Subsolar (noon) position
    r_m_noon, r_noon = get_position(alt_m, subsolar_lon)
    # Anti-subsolar (midnight) position
    r_m_midnight, r_midnight = get_position(alt_m, anti_subsolar_lon)
    
    # Velocity vectors (circular orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)  # Magnitude
    v_vec_noon = np.array([0, v_mag, 0])  # Simplified - would need proper transformation
    v_vec_midnight = np.array([0, -v_mag, 0])  # Opposite direction
    
    # Calculate radial and along-track directions
    radial_dir_noon = r_m_noon / np.linalg.norm(r_m_noon)
    v_perp_noon = v_vec_noon - np.dot(v_vec_noon, radial_dir_noon) * radial_dir_noon
    along_track_dir_noon = v_perp_noon / np.linalg.norm(v_perp_noon) if np.linalg.norm(v_perp_noon) > 0 else np.array([1,0,0])
    
    radial_dir_midnight = r_m_midnight / np.linalg.norm(r_m_midnight)
    v_perp_midnight = v_vec_midnight - np.dot(v_vec_midnight, radial_dir_midnight) * radial_dir_midnight
    along_track_dir_midnight = v_perp_midnight / np.linalg.norm(v_perp_midnight) if np.linalg.norm(v_perp_midnight) > 0 else np.array([1,0,0])
    
    for angle_deg, tether_angle in zip([0, 5, 45, 85, 90], TETHER_ANGLES):
        # Tether direction (angle degrees off radial)
        tether_dir_noon = (np.cos(tether_angle) * radial_dir_noon + 
                          np.sin(tether_angle) * along_track_dir_noon)
        tether_dir_midnight = (np.cos(tether_angle) * radial_dir_midnight + 
                             np.sin(tether_angle) * along_track_dir_midnight)
        
        for condition, r_m, r, v_vec, tether_dir in [
            ('noon', r_m_noon, r_noon, v_vec_noon, tether_dir_noon),
            ('midnight', r_m_midnight, r_midnight, v_vec_midnight, tether_dir_midnight)
        ]:
            # Magnetic field
            B = igrf_cartesian(obs_time.datetime, r) * 1e-9  # Convert nT to T
            B_mag = np.linalg.norm(B)
            results[angle_deg][condition]['B_field'].append(B_mag)
            
            # Plasma density and SZA
            n, sza = calculate_plasma_density(r_m, obs_time.datetime)
            results[angle_deg][condition]['plasma_density'].append(n)
            results[angle_deg][condition]['sza'].append(sza)
            
            # Atmospheric density
            rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime, f107=75.0, f107a=75.0)
            results[angle_deg][condition]['atmos_density'].append(rho)
            
            # Velocity
            results[angle_deg][condition]['velocity'].append(v_mag)
            
            # Vemf (full tether length)
            v_cross_B = np.cross(v_vec, B[:,0])
            Vemf_total = L * v_cross_B
            Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
            results[angle_deg][condition]['Vemf'].append(np.linalg.norm(Vemf_proj))
            
            # Current
            dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
            I_mag = dIdl * np.sign(Vemf_proj)  # 1
            results[angle_deg][condition]['current'].append(abs(I_mag))
            
            # Drag force
            tether_projected_area = L * w * np.abs(np.cos(tether_angle))
            total_area = Area1 + tether_projected_area
            F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
            results[angle_deg][condition]['drag_force'].append(F_drag_mag)
            
            # Lorentz force
            F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))  #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
            results[angle_deg][condition]['lorentz_force'].append(F_lorentz)

# Define colors and line styles for different angles
colors = ['blue', 'green', 'red', 'orange', 'purple']
angle_labels = ['0°', '5°', '45°', '85°', '90°']

# Create the plots with selected tether angles
plt.figure(figsize=(16, 12))

# 1. All angles at Midnight
plt.subplot(2, 2, 1)
for i, angle_deg in enumerate([0, 5, 45, 85, 90]):
    plt.semilogx(results[angle_deg]['midnight']['drag_force'], altitudes_km, 
                 color=colors[i], linestyle='-', label=f'Drag {angle_labels[i]}')
    plt.semilogx(results[angle_deg]['midnight']['lorentz_force'], altitudes_km, 
                 color=colors[i], linestyle='--', label=f'Lorentz {angle_labels[i]}')
plt.xlabel('Force (N)')
plt.ylabel('Altitude (km)')
plt.title('All Forces at Midnight')
plt.grid(True)
plt.legend(loc='upper right', fontsize=8)

# 2. All angles at Noon
plt.subplot(2, 2, 2)
for i, angle_deg in enumerate([0, 5, 45, 85, 90]):
    plt.semilogx(results[angle_deg]['noon']['drag_force'], altitudes_km, 
                 color=colors[i], linestyle='-', label=f'Drag {angle_labels[i]}')
    plt.semilogx(results[angle_deg]['noon']['lorentz_force'], altitudes_km, 
                 color=colors[i], linestyle='--', label=f'Lorentz {angle_labels[i]}')
plt.xlabel('Force (N)')
plt.ylabel('Altitude (km)')
plt.title('All Forces at Noon')
plt.grid(True)
plt.legend(loc='upper right', fontsize=8)

# 3. Force Ratio Comparison for selected angles
plt.subplot(2, 2, 3)
selected_angles = [0, 45, 90]
angle_labels = ['0°', '45°', '90°']
for i, angle_deg in enumerate(selected_angles):
    ratio_noon = np.array(results[angle_deg]['noon']['lorentz_force']) / np.array(results[angle_deg]['noon']['drag_force'])
    ratio_midnight = np.array(results[angle_deg]['midnight']['lorentz_force']) / np.array(results[angle_deg]['midnight']['drag_force'])
    plt.semilogx(ratio_noon, altitudes_km, color=colors[selected_angles.index(angle_deg)], 
                 linestyle='-', label=f'{angle_labels[selected_angles.index(angle_deg)]} Noon')
    plt.semilogx(ratio_midnight, altitudes_km, color=colors[selected_angles.index(angle_deg)], 
                 linestyle='--', label=f'{angle_labels[selected_angles.index(angle_deg)]} Midnight')
plt.axvline(x=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Force Ratio (Lorentz/Drag)')
plt.ylabel('Altitude (km)')
plt.title('Lorentz to Drag Force Ratio\n(Selected Angles)')
plt.grid(True)
plt.legend(loc='upper right', fontsize=8)

# 4. Bar graph for forces at 500km altitude
plt.subplot(2, 2, 4)

# Find index closest to 500km
alt_500km_idx = np.argmin(np.abs(altitudes_km - 500))

# Prepare data for plotting
angles = [0, 5, 45, 85, 90]
drag_forces_noon = []
lorentz_forces_noon = []
drag_forces_midnight = []
lorentz_forces_midnight = []

for angle_deg in angles:
    drag_forces_noon.append(results[angle_deg]['noon']['drag_force'][alt_500km_idx])
    lorentz_forces_noon.append(results[angle_deg]['noon']['lorentz_force'][alt_500km_idx])
    drag_forces_midnight.append(results[angle_deg]['midnight']['drag_force'][alt_500km_idx])
    lorentz_forces_midnight.append(results[angle_deg]['midnight']['lorentz_force'][alt_500km_idx])

# Create bar plot
x_pos = np.arange(len(angles))
width = 0.35

plt.bar(x_pos - width/2, drag_forces_noon, width, label='Drag Noon', alpha=0.8, color='blue')
plt.bar(x_pos - width/2, lorentz_forces_noon, width, bottom=drag_forces_noon, 
        label='Lorentz Noon', alpha=0.8, color='lightblue')
plt.bar(x_pos + width/2, drag_forces_midnight, width, label='Drag Midnight', alpha=0.8, color='red')
plt.bar(x_pos + width/2, lorentz_forces_midnight, width, bottom=drag_forces_midnight, 
        label='Lorentz Midnight', alpha=0.8, color='lightcoral')

plt.xlabel('Tether Angle (degrees)')
plt.ylabel('Force (N)')
plt.title('Drag and Lorentz Forces at 500km Altitude\nby Tether Orientation')
plt.xticks(x_pos, ['0°', '5°', '45°', '85°', '90°'])
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Animations w/ Impact Included

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0005 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 0  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

#blue: #0072B2   orange: #E69F00 


# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Altitude decay rate (vertical)

import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.01  # longer run

# Constants
EARTH_RADIUS_KM = 6378.0

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 0  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)
Area2 = Area1 + CSST_Area # Tethered Area

projected_area = Area2 * np.cos(TETHER_ANGLE_RAD)

sat.SetField("DragArea", projected_area)
sat.SetField("SRPArea", projected_area)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only process every Nth frame

# Storage for altitude data only
time_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative values
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store altitude data only
print("Computing orbital decay rate...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    r_m = r_km * 1000
    
    # Calculate altitude
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    
    # Store data for decay rate calculation
    time_data.append(i)
    altitude_data.append(altitude_m)

# Convert to numpy arrays
time_array = np.array(time_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

In [ ]:
# Altitude decay rate (80 degrees)

import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.01  # longer run

# Constants
EARTH_RADIUS_KM = 6378.0

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 80  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)
Area2 = Area1 + CSST_Area # Tethered Area

projected_area = Area2 * np.cos(TETHER_ANGLE_RAD)

sat.SetField("DragArea", projected_area)
sat.SetField("SRPArea", projected_area)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only process every Nth frame

# Storage for altitude data only
time_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative values
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store altitude data only
print("Computing orbital decay rate...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    r_m = r_km * 1000
    
    # Calculate altitude
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    
    # Store data for decay rate calculation
    time_data.append(i)
    altitude_data.append(altitude_m)

# Convert to numpy arrays
time_array = np.array(time_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

TSS-1R

In [ ]:
# Reset to TSS-1R LEO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6678)   
sat.SetField("RadApo", 6678.005)
sat.SetField("INC", 28.5)
sat.SetField("RAAN", 0)
sat.SetField("AOP", 0)
sat.SetField("TA", 0) 

CSST_Mass = 0.083      # CSST = 0.083kg    TSS-1R: ?
w = 0.00254            # CCST = 0.075m     TSS-1R: 0.00254
L = 19695              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L  


# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Combined Altitude and Longitude Profiles Analysis (5° Off-Vertical Tether Orientation)
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
TETHER_ANGLE = np.radians(5)  # 5 degrees off vertical (radians)

# Configuration - Flexible altitude values for longitude profiles
LONGITUDE_ALTITUDES = [300, 700]  # km - can be modified

# Single observation time (converted to Astropy Time)
obs_time = Time(datetime(2025, 3, 20, 12, 0, 0))  # Fixed universal time (Equinox)

# Find subsolar and anti-subsolar points at this time
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg  # Subsolar longitude
anti_subsolar_lon = (subsolar_lon + 180) % 360  # Opposite side

# Altitude range (100-1000 km) for altitude profiles
altitudes_km = np.linspace(100, 1000, 50)
altitudes_m = altitudes_km * 1000 + R_EARTH

# Longitude range (0-360° in 5° steps) for longitude profiles
longitudes = np.linspace(0, 360, 73)  # 5° resolution

# Initialize arrays for storage
results_altitude = {
    'noon': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    },
    'midnight': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }
}

results_longitude = {}
for alt in LONGITUDE_ALTITUDES:
    results_longitude[f"{alt}km"] = {
        'longitude': [], 'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }

def get_position(alt_m, lon_deg):
    """Get position vector for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    return np.array([x, y, z]), np.array([x/1000, y/1000, z/1000])  # Return both m and km versions

def get_position_and_tether_dir(alt_m, lon_deg):
    """Get position vector and tether direction for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    
    # Position vectors (m and km)
    r_m = np.array([x, y, z])
    r_km = r_m / 1000
    
    # Calculate directions
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Velocity vector (tangential to orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)
    v_vec = np.array([-v_mag*np.sin(np.radians(lon_deg)), 
                      v_mag*np.cos(np.radians(lon_deg)), 
                      0])
    
    # Calculate along-track direction
    v_perp = v_vec - np.dot(v_vec, radial_dir) * radial_dir
    along_track_dir = v_perp / np.linalg.norm(v_perp) if np.linalg.norm(v_perp) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir = np.cos(TETHER_ANGLE) * radial_dir + np.sin(TETHER_ANGLE) * along_track_dir
    
    return r_m, r_km, tether_dir, v_vec

# Calculate altitude profiles
print("Calculating altitude profiles...")
for alt_m, alt_km in zip(altitudes_m, altitudes_km):
    # Subsolar (noon) position
    r_m_noon, r_noon = get_position(alt_m, subsolar_lon)
    # Anti-subsolar (midnight) position
    r_m_midnight, r_midnight = get_position(alt_m, anti_subsolar_lon)
    
    # Velocity vectors (circular orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)  # Magnitude
    v_vec_noon = np.array([0, v_mag, 0])  # Simplified - would need proper transformation
    v_vec_midnight = np.array([0, -v_mag, 0])  # Opposite direction
    
    # Calculate radial and along-track directions
    radial_dir_noon = r_m_noon / np.linalg.norm(r_m_noon)
    v_perp_noon = v_vec_noon - np.dot(v_vec_noon, radial_dir_noon) * radial_dir_noon
    along_track_dir_noon = v_perp_noon / np.linalg.norm(v_perp_noon) if np.linalg.norm(v_perp_noon) > 0 else np.array([1,0,0])
    
    radial_dir_midnight = r_m_midnight / np.linalg.norm(r_m_midnight)
    v_perp_midnight = v_vec_midnight - np.dot(v_vec_midnight, radial_dir_midnight) * radial_dir_midnight
    along_track_dir_midnight = v_perp_midnight / np.linalg.norm(v_perp_midnight) if np.linalg.norm(v_perp_midnight) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir_noon = (np.cos(TETHER_ANGLE) * radial_dir_noon + 
                      np.sin(TETHER_ANGLE) * along_track_dir_noon)
    tether_dir_midnight = (np.cos(TETHER_ANGLE) * radial_dir_midnight + 
                         np.sin(TETHER_ANGLE) * along_track_dir_midnight)
    
    for condition, r_m, r, v_vec, tether_dir in [
        ('noon', r_m_noon, r_noon, v_vec_noon, tether_dir_noon),
        ('midnight', r_m_midnight, r_midnight, v_vec_midnight, tether_dir_midnight)
    ]:
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_altitude[condition]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_altitude[condition]['plasma_density'].append(n)
        results_altitude[condition]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results_altitude[condition]['atmos_density'].append(rho)
        
        # Velocity
        results_altitude[condition]['velocity'].append(v_mag)
        
        # Vemf (full tether length)
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_altitude[condition]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)        # F prop L
        results_altitude[condition]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_altitude[condition]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_altitude[condition]['lorentz_force'].append(F_lorentz)

# Calculate longitude profiles
print("Calculating longitude profiles...")
for alt_km in LONGITUDE_ALTITUDES:
    alt_m = alt_km * 1000 + R_EARTH
    key = f"{alt_km}km"
    
    for lon in longitudes:
        r_m, r_km, tether_dir, v_vec = get_position_and_tether_dir(alt_m, lon)
        v_mag = np.linalg.norm(v_vec)
        
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r_km) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results_longitude[key]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results_longitude[key]['plasma_density'].append(n)
        results_longitude[key]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results_longitude[key]['atmos_density'].append(rho)
        
        # Velocity
        results_longitude[key]['velocity'].append(v_mag)
        
        # Vemf
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results_longitude[key]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)
        results_longitude[key]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results_longitude[key]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0]))
        results_longitude[key]['lorentz_force'].append(F_lorentz)
        
        # Longitude
        results_longitude[key]['longitude'].append(lon)

# NEW FUNCTIONALITY: Print values at specified altitude
def print_values_at_altitude(altitude_km):
    """Print all parameter values at a specified altitude"""
    # Find the closest altitude in our data
    idx = np.argmin(np.abs(altitudes_km - altitude_km))
    actual_altitude = altitudes_km[idx]
    
    print(f"\n{'='*80}")
    print(f"PARAMETER VALUES AT {actual_altitude:.1f} km ALTITUDE")
    print(f"{'='*80}")
    
    print(f"{'Parameter':<25} {'Noon':<15} {'Midnight':<15} {'Units':<10}")
    print(f"{'-'*80}")
    
    # Magnetic Field
    print(f"{'Magnetic Field':<25} {results_altitude['noon']['B_field'][idx]:<15.2e} {results_altitude['midnight']['B_field'][idx]:<15.2e} {'T':<10}")
    
    # Plasma Density
    print(f"{'Plasma Density':<25} {results_altitude['noon']['plasma_density'][idx]:<15.2e} {results_altitude['midnight']['plasma_density'][idx]:<15.2e} {'m⁻³':<10}")
    
    # Atmospheric Density
    print(f"{'Atmospheric Density':<25} {results_altitude['noon']['atmos_density'][idx]:<15.2e} {results_altitude['midnight']['atmos_density'][idx]:<15.2e} {'kg/m³':<10}")
    
    # Velocity
    print(f"{'Orbital Velocity':<25} {results_altitude['noon']['velocity'][idx]:<15.2f} {results_altitude['midnight']['velocity'][idx]:<15.2f} {'m/s':<10}")
    
    # Induced EMF
    print(f"{'Induced EMF (Vemf)':<25} {results_altitude['noon']['Vemf'][idx]:<15.2f} {results_altitude['midnight']['Vemf'][idx]:<15.2f} {'V':<10}")
    
    # Current
    print(f"{'Tether Current':<25} {results_altitude['noon']['current'][idx]:<15.2e} {results_altitude['midnight']['current'][idx]:<15.2e} {'A':<10}")
    
    # Drag Force
    print(f"{'Drag Force':<25} {results_altitude['noon']['drag_force'][idx]:<15.2e} {results_altitude['midnight']['drag_force'][idx]:<15.2e} {'N':<10}")
    
    # Lorentz Force
    print(f"{'Lorentz Force':<25} {results_altitude['noon']['lorentz_force'][idx]:<15.2e} {results_altitude['midnight']['lorentz_force'][idx]:<15.2e} {'N':<10}")
    
    # SZA
    print(f"{'Solar Zenith Angle':<25} {results_altitude['noon']['sza'][idx]:<15.1f} {results_altitude['midnight']['sza'][idx]:<15.1f} {'°':<10}")
    
    # Force Ratio
    ratio_noon = results_altitude['noon']['lorentz_force'][idx] / results_altitude['noon']['drag_force'][idx]
    ratio_midnight = results_altitude['midnight']['lorentz_force'][idx] / results_altitude['midnight']['drag_force'][idx]
    print(f"{'Lorentz/Drag Ratio':<25} {ratio_noon:<15.2f} {ratio_midnight:<15.2f} {'':<10}")
    
    # Net Force
    net_noon = results_altitude['noon']['lorentz_force'][idx] - results_altitude['noon']['drag_force'][idx]
    net_midnight = results_altitude['midnight']['lorentz_force'][idx] - results_altitude['midnight']['drag_force'][idx]
    print(f"{'Net Force (L-D)':<25} {net_noon:<15.2e} {net_midnight:<15.2e} {'N':<10}")
    
    print(f"{'='*80}")

# Print values at 300 km
print_values_at_altitude(300)

# Create comprehensive figure with larger fonts
plt.figure(figsize=(25, 30))
plt.suptitle(f'Electrodynamic Tether Analysis (5° Off-Vertical Orientation)\n'
             f'Observation Time: {obs_time.datetime.strftime("%Y-%m-%d %H:%M UTC")}, '
             f'Subsolar Longitude: {subsolar_lon:.1f}°',
             fontsize=20, fontweight='bold', y=1.02)


# Set global font sizes
label_fontsize = 19
title_fontsize = 22
legend_fontsize = 16
tick_fontsize = 16

# Vertical lines for subsolar/anti-subsolar points
subsol_line = {'ls': '--', 'color': 'gold', 'alpha': 0.7, 'label': 'Subsolar'}
antisol_line = {'ls': '--', 'color': 'navy', 'alpha': 0.7, 'label': 'Anti-subsolar'}

# Row 1-2: Altitude Profiles (8 subplots)
# 1. Magnetic Field vs Altitude
plt.subplot(4, 4, 1)
plt.plot(results_altitude['midnight']['B_field'], altitudes_km, label='Midnight (Anti-subsolar)')
plt.plot(results_altitude['noon']['B_field'], altitudes_km, label='Noon (Subsolar)')
plt.xlabel('Magnetic Field Strength (T)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(a) Magnetic Field vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 2. Plasma Density vs Altitude
plt.subplot(4, 4, 2)
plt.semilogx(results_altitude['midnight']['plasma_density'], altitudes_km, 
             label=f'Midnight (SZA={results_altitude["midnight"]["sza"][0]:.1f}°)')
plt.semilogx(results_altitude['noon']['plasma_density'], altitudes_km, 
             label=f'Noon (SZA={results_altitude["noon"]["sza"][0]:.1f}°)')
plt.xlabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(b) Plasma Density vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 3. Atmospheric Density vs Altitude
plt.subplot(4, 4, 3)
plt.semilogx(results_altitude['midnight']['atmos_density'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['atmos_density'], altitudes_km, label='Noon')
plt.xlabel('Atmospheric Density (kg/m$^3$)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(c) Atmospheric Density vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 4. Orbital Velocity vs Altitude
plt.subplot(4, 4, 4)
plt.plot(results_altitude['midnight']['velocity'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['velocity'], altitudes_km, label='Noon')
plt.xlabel('Orbital Velocity (m/s)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(d) Circular Orbit Velocity vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 5. Induced EMF vs Altitude
plt.subplot(4, 4, 5)
plt.plot(results_altitude['midnight']['Vemf'], altitudes_km, label='Midnight')
plt.plot(results_altitude['noon']['Vemf'], altitudes_km, label='Noon')
plt.xlabel('Induced EMF (V)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(e) Tether EMF vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 6. Current vs Altitude
plt.subplot(4, 4, 6)
plt.semilogx(results_altitude['midnight']['current'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['current'], altitudes_km, label='Noon')
plt.xlabel('Current (A)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(f) Tether Current vs Altitude\n(Dominated by Plasma Density)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 7. Drag Force vs Altitude
plt.subplot(4, 4, 7)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, label='Noon')
plt.xlabel('Drag Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(g) Atmospheric Drag vs Altitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 8. Lorentz Force vs Altitude
plt.subplot(4, 4, 8)
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, label='Midnight')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, label='Noon')
plt.xlabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(h) Electrodynamic Tether Force vs Altitude\n(Day-Night Difference)', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 3: Longitude Profiles (4 key parameters)
# 9. Magnetic Field vs Longitude
plt.subplot(4, 4, 9)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.plot(results_longitude[key]['longitude'], results_longitude[key]['B_field'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Magnetic Field (T)', fontsize=label_fontsize)
plt.title('(i) B-field vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 10. Plasma Density vs Longitude
plt.subplot(4, 4, 10)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['plasma_density'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Plasma Density (m$^{-3}$)', fontsize=label_fontsize)
plt.title('(j) Plasma Density vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 11. Drag Force vs Longitude
plt.subplot(4, 4, 11)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['drag_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Drag Force (N)', fontsize=label_fontsize)
plt.title('(k) Atmospheric Drag vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 12. Lorentz Force vs Longitude
plt.subplot(4, 4, 12)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    plt.semilogy(results_longitude[key]['longitude'], results_longitude[key]['lorentz_force'], label=f'{alt} km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Lorentz Force (N)', fontsize=label_fontsize)
plt.title('(l) Electrodynamic Tether Force vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# Row 4: Force Comparisons and Ratios
# 13. Altitude: Drag vs Lorentz Force Comparison (Noon)
plt.subplot(4, 4, 13)
plt.semilogx(results_altitude['noon']['drag_force'], altitudes_km, 'r-', label='Drag (Noon)')
plt.semilogx(results_altitude['noon']['lorentz_force'], altitudes_km, 'r--', label='Lorentz (Noon)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(m) Drag vs Lorentz Force (Noon)', fontsize=title_fontsize)
plt.grid(True)
plt.xlim(1e-5,1e4)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 14. Altitude: Drag vs Lorentz Force Comparison (Midnight)
plt.subplot(4, 4, 14)
plt.semilogx(results_altitude['midnight']['drag_force'], altitudes_km, 'b-', label='Drag (Midnight)')
plt.semilogx(results_altitude['midnight']['lorentz_force'], altitudes_km, 'b--', label='Lorentz (Midnight)')
plt.xlabel('Force (N)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(n) Drag vs Lorentz Force (Midnight)', fontsize=title_fontsize)
plt.grid(True)
plt.xlim(1e-5,1e4)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 15. Altitude: Force Ratio
plt.subplot(4, 4, 15)
ratio_noon = np.array(results_altitude['noon']['lorentz_force']) / np.array(results_altitude['noon']['drag_force'])
ratio_midnight = np.array(results_altitude['midnight']['lorentz_force']) / np.array(results_altitude['midnight']['drag_force'])
plt.semilogx(ratio_midnight, altitudes_km, label='Midnight (Lorentz/Drag)') # 'm-', 
plt.semilogx(ratio_noon, altitudes_km, label='Noon (Lorentz/Drag)') #'g-',
plt.axvline(x=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.ylabel('Altitude (km)', fontsize=label_fontsize)
plt.title('(o) Lorentz to Drag Force Ratio', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

# 16. Longitude: Force Ratio
plt.subplot(4, 4, 16)
for alt in LONGITUDE_ALTITUDES:
    key = f"{alt}km"
    ratio = np.array(results_longitude[key]['lorentz_force']) / np.array(results_longitude[key]['drag_force'])
    plt.semilogy(results_longitude[key]['longitude'], ratio, label=f'{alt}km (Lorentz/Drag)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Longitude (°)', fontsize=label_fontsize)
plt.ylabel('Force Ratio (Lorentz/Drag)', fontsize=label_fontsize)
plt.title('(p) Lorentz to Drag Force Ratio vs Longitude', fontsize=title_fontsize)
plt.grid(True)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

GTOSat

In [ ]:
# Reset to GTOSat GTO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6563)   
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 0  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Reset to GTOSat GTO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6563)   
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included (45)

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = -45  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Reset to GTOSat GTO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6563)   
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included (90)

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = -90  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Reset to GTO Orbit (300km perigee)

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6683)   # 6563(180) + 120
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included (Vertical)

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = 0  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Reset to GTO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6683)   # 6563(180) + 120
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included (45)

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = -45  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

In [ ]:
# Reset to GTO Orbit

sat.SetField("DateFormat", "UTCGregorian")          #"12 Mar 2020 15:00:00.000"
sat.SetField("Epoch", "13 Oct 2021 12:00:00.000")
sat.SetField("CoordinateSystem", "EarthMJ2000Eq")
sat.SetField("DisplayStateType", "ModifiedKeplerian")
sat.SetField("RadPer", 6683)   # 6563(180) + 120
sat.SetField("RadApo", 42165) 
sat.SetField("INC", 27) 
sat.SetField("RAAN", 0)
sat.SetField("AOP", 180)                            
sat.SetField("TA", 0)

CSST_Mass = 0.083    # CSST = 0.083kg    TSS-1R: ?
w = 0.075            # CCST = 0.075m     TSS-1R: 0.005
L = 20              # CSST = 20m        TSS-1R: 19695
CSST_Area = (2 / np.pi) * w * L

# Perform top level initialization
gmat.Initialize()
# Perform the integation subsysem initialization
pdprop.PrepareInternals()

In [ ]:
# Animations w/ Impact Included (90)

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import HTML
from datetime import datetime, timedelta
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from tqdm import tqdm
from scipy import stats  # For linear regression

Years = 0.0025 #short

# Set up the figure with a grid layout
plt.rcParams['figure.figsize'] = (20, 14)  # Increased size for additional plot
plt.rcParams['animation.embed_limit'] = 1000000  # Increase animation size limit to 100MB
fig = plt.figure(constrained_layout=True)
gs = GridSpec(4, 2, figure=fig)  # Changed to 4 rows

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])  # Position plot
ax2 = fig.add_subplot(gs[1, 0])  # Force plot
ax3 = fig.add_subplot(gs[2, 0])  # Delta-V plot
ax4 = fig.add_subplot(gs[3, 0])  # Impact plot (new)
ax5 = fig.add_subplot(gs[:, 1], projection='3d')  # 3D vector plot

plt.close()  # Prevents duplicate display in notebook

# Constants
EARTH_RADIUS_KM = 6378.0
ALTITUDE_THRESHOLD_KM = 100.0

# Tether parameters for impact calculation
TETHER_DIAMETER = w  # w meters (2mm diameter)
TETHER_LENGTH = L  # L meters (use the same L as defined elsewhere)
CSST_Area = (2 / np.pi) * w * L  # Area = width x length x 2/pi (random twisting factor)   Link: https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=5080&context=smallsat
Area1 = 0.105   # CubeSat Area (m^2) (from Antoine's slides, assuming random tumbling)  0.076 for old version
Area2 = Area1 + CSST_Area # Tethered Area

# Tether angle parameter (in degrees)
TETHER_ANGLE_DEG = -90  # Example value - adjust as needed
TETHER_ANGLE_RAD = np.radians(TETHER_ANGLE_DEG)

# Simulation parameters
start_date = datetime(2020, 11, 20, 12, 0, 0)
Time_step = 60.0  # seconds
total_steps = int(525600 * Years)  # 525600 minutes in a year
sample_rate = 10  # Only plot every Nth frame

# Storage for all data
frame_data = []
time_data = []
pos_data = []
lorentz_data = []
drag_data = []
dv_lorentz_data = []
dv_drag_data = []
date_data = []
impact_rate_data = []
cumulative_impacts_data = []
altitude_data = []  # Store altitude for decay rate calculation

# Initialize cumulative delta-V and impacts
cumulative_dv_lorentz = 0.0
cumulative_dv_drag = 0.0
cumulative_impacts = 0.0

# Pre-propagate and store all frame data
print("Pre-computing animation frames...")
for i in tqdm(range(0, total_steps, sample_rate)):
    # Propagate to current step
    if i > 0:
        gator.Step(Time_step * sample_rate)
    
    # Get current state
    gatorstate = gator.GetState()
    r_km = np.array([gatorstate[j] for j in range(3)])
    v_kms = np.array([gatorstate[j+3] for j in range(3)])
    
    # Calculate time and vectors
    current_date = start_date + timedelta(minutes=i)
    B_nT = igrf_cartesian(current_date, r_km)
    B_T = B_nT * 1e-9
    r_m = r_km * 1000
    v_ms = v_kms * 1000
    
    # Calculate radial direction
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Calculate velocity direction
    velocity_dir = v_ms / np.linalg.norm(v_ms)
    
    # Calculate tether direction in orbital plane (opposite to velocity direction)
    # The tether direction is offset from radial by theta in the opposite direction of velocity
    orbital_plane_normal = np.cross(radial_dir, velocity_dir)
    orbital_plane_normal = orbital_plane_normal / np.linalg.norm(orbital_plane_normal)
    
    # Create a rotation matrix to rotate radial vector around orbital plane normal
    cos_theta = np.cos(TETHER_ANGLE_RAD)
    sin_theta = np.sin(TETHER_ANGLE_RAD)
    
    # Rodrigues' rotation formula
    tether_dir = (radial_dir * cos_theta + 
                 np.cross(orbital_plane_normal, radial_dir) * sin_theta +
                 orbital_plane_normal * np.dot(orbital_plane_normal, radial_dir) * (1 - cos_theta))
    
    # Ensure tether direction is unit vector
    tether_dir = tether_dir / np.linalg.norm(tether_dir)
    
    # Calculate all relevant vectors
    v_cross_B = np.cross(v_ms, B_T[:,0])
    total_Vemf = L * v_cross_B
    Vemf_proj = np.dot(total_Vemf, tether_dir) * tether_dir
    I_mag = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))   # Use for F prop L
    dIdl = I_func(r_m, current_date, np.linalg.norm(Vemf_proj))
    #I_mag = L * dIdl                                               # Use for F prop L^2
    #I_mag = 1                                                       # Use for TSS-1R
    I_vec = np.sign(np.dot(total_Vemf, tether_dir)) * I_mag * tether_dir
    F_lorentz = L * np.cross(I_vec, B_T[:,0]) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
    
    # Calculate drag force
    altitude_m = np.linalg.norm(r_m) - (EARTH_RADIUS_KM * 1000)
    rho = atmospheric_density_nrlmsise00(r_m, current_date, 90, 90)[0]   # F10.7 = 90 Source: https://www.swpc.noaa.gov/products/solar-cycle-progression 
    v_mag_ms = np.linalg.norm(v_ms)
    
    # Calculate projected area for drag (L*cos(theta) projected onto radial direction)
    radial_projected_length = L * np.abs(np.dot(tether_dir, radial_dir))
    projected_area = TETHER_DIAMETER * radial_projected_length + Area1
    
    F_drag = -0.5 * Cd1 * projected_area * rho * v_mag_ms * v_ms     
    
    # Calculate delta-V contributions
    dt = Time_step * sample_rate
    dv_lorentz = (F_lorentz / Mass2) * dt  # m/s
    dv_drag = (F_drag / Mass2) * dt  # m/s
    
    # Update cumulative delta-V (convert to km/s)
    cumulative_dv_lorentz += np.linalg.norm(dv_lorentz) / 1000
    cumulative_dv_drag += np.linalg.norm(dv_drag) / 1000
    
    # Calculate debris impact rate with projected area
    debris_dens = debris_density(r_m)  # particles/m^3
    
    # Set the projected area for the satellite model
    sat.SetField("DragArea", projected_area)
    sat.SetField("SRPArea", projected_area)
    
    impact_rate = debris_dens * v_mag_ms * projected_area  # impacts/second
    
    # Update cumulative impacts
    cumulative_impacts += impact_rate * dt
    
    # Store all data
    frame_data.append({
        'r_km': r_km,
        'v_ms': v_ms,
        'B_T': B_T,
        'total_Vemf': total_Vemf,
        'Vemf_proj': Vemf_proj,
        'I_vec': I_vec,
        'F_lorentz': F_lorentz,
        'F_drag': F_drag,
        'date': current_date,
        'altitude': np.linalg.norm(r_km) - EARTH_RADIUS_KM,
        'tether_dir': tether_dir,
        'radial_dir': radial_dir,
        'velocity_dir': velocity_dir,
        'impact_rate': impact_rate,
        'cumulative_impacts': cumulative_impacts,
        'projected_area': projected_area
    })
    
    time_data.append(i)
    pos_data.append(r_km)
    lorentz_data.append(np.linalg.norm(F_lorentz))
    drag_data.append(np.linalg.norm(F_drag))
    dv_lorentz_data.append(cumulative_dv_lorentz)
    dv_drag_data.append(cumulative_dv_drag)
    date_data.append(current_date)
    impact_rate_data.append(impact_rate)
    cumulative_impacts_data.append(cumulative_impacts)
    altitude_data.append(altitude_m)  # Store altitude in meters

# Convert to numpy arrays
time_array = np.array(time_data)
pos_array = np.array(pos_data)
lorentz_array = np.array(lorentz_data)
drag_array = np.array(drag_data)
dv_lorentz_array = np.array(dv_lorentz_data)
dv_drag_array = np.array(dv_drag_data)
impact_rate_array = np.array(impact_rate_data)
cumulative_impacts_array = np.array(cumulative_impacts_data)
altitude_array = np.array(altitude_data)

# Calculate orbital decay rate using linear regression
# Convert time from minutes to days for decay rate in meters/day
time_days = time_array / (60 * 24)  # Convert minutes to days

# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(time_days, altitude_array)
decay_rate_m_per_day = -slope  # Negative because altitude decreases over time

print(f"Orbital decay rate: {decay_rate_m_per_day:.4f} meters/day")
print(f"R-squared value: {r_value**2:.4f}")

# Find maximum magnitudes for scaling
max_v = max(np.linalg.norm(data['v_ms']) for data in frame_data)
max_B = max(np.linalg.norm(data['B_T'][:,0]) for data in frame_data)
max_Vemf = max(np.linalg.norm(data['total_Vemf']) for data in frame_data)
max_Vemf_proj = max(np.linalg.norm(data['Vemf_proj']) for data in frame_data)
max_I = max(np.linalg.norm(data['I_vec']) for data in frame_data)
max_F = max(np.linalg.norm(data['F_lorentz']) for data in frame_data)

# Create initial plots (before animation)
# Position plot
ax1.plot(date_data, pos_array[:,0], label='X Position')
ax1.plot(date_data, pos_array[:,1], label='Y Position')
ax1.plot(date_data, pos_array[:,2], label='Z Position')
ax1.set_ylabel('Position (km)')
ax1.set_title('Orbit Position Components')
ax1.legend()
ax1.grid(True)
pos_vline = ax1.axvline(date_data[0], color='r', linestyle='--')

# Force plot
ax2.semilogy(date_data, lorentz_array, label='Lorentz Force', color='#0072B2')
ax2.semilogy(date_data, drag_array, label='Drag Force', color='#E69F00')
ax2.set_ylabel('Force Magnitude (N)')
ax2.set_title('Forces Acting on Satellite')
ax2.legend()
ax2.grid(True)
force_vline = ax2.axvline(date_data[0], color='r', linestyle='--')

# Delta-V plot
ax3.plot(date_data, dv_lorentz_array, label='Lorentz ΔV', color='#0072B2')
ax3.plot(date_data, dv_drag_array, label='Drag ΔV', color='#E69F00')
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative ΔV (km/s)')
ax3.set_title('Cumulative Delta-V Contributions')
ax3.legend()
ax3.grid(True)
dv_vline = ax3.axvline(date_data[0], color='r', linestyle='--')

# Impact plot (new)
ax4.semilogy(date_data, impact_rate_array, label='Impact Rate', color='orange', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(date_data, cumulative_impacts_array, label='Cumulative Impacts', color='green', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Impact Rate (impacts/s)', color='orange')
ax4_twin.set_ylabel('Cumulative Impacts', color='green')
ax4.set_title('Debris Impact Statistics')
ax4.grid(True)
impact_vline = ax4.axvline(date_data[0], color='r', linestyle='--')

# Format x-axes
for ax in [ax1, ax2, ax3, ax4]:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def update(frame):
    """Update function for animation with dynamic vector scaling"""
    data = frame_data[frame]
    current_date = data['date']
    
    # Update 3D vector plot
    ax5.clear()
    
    # Base scale for visualization
    base_scale = 2.0
    
    # Calculate relative scaling factors (normalized to max values)
    scale_factors = {
        'v': base_scale * (np.linalg.norm(data['v_ms']) / max_v),
        'B': base_scale * (np.linalg.norm(data['B_T'][:,0]) / max_B),
        'total_Vemf': base_scale * (np.linalg.norm(data['total_Vemf']) / max_Vemf),
        'Vemf_proj': base_scale * (np.linalg.norm(data['Vemf_proj']) / max_Vemf_proj),
        'I': base_scale * (np.linalg.norm(data['I_vec']) / max_I),
        'F': base_scale * (np.linalg.norm(data['F_lorentz']) / max_F),
        'tether': 1.0,  # Tether direction is unit vector
        'radial': 1.0,  # Radial direction is unit vector
        'velocity': 1.0  # Velocity direction is unit vector
    }

    # Calculate consistent scaling for Vemf vectors
    vemf_total_mag = np.linalg.norm(data['total_Vemf'])
    if vemf_total_mag > 0:
        vemf_scale = base_scale / vemf_total_mag
    else:
        vemf_scale = 0
    
    # Plot vectors with dynamic scaling
    vectors = [
        (data['v_ms']/np.linalg.norm(data['v_ms'])*scale_factors['v'], 'Velocity', 'blue'),
        (data['B_T'][:,0]/np.linalg.norm(data['B_T'][:,0])*scale_factors['B'], 'B-field', 'red'),
        (data['total_Vemf']*vemf_scale, 'Total Vemf', 'green'),
        (data['Vemf_proj']*vemf_scale, 'Usable Vemf', 'cyan'),
        (data['I_vec']/np.linalg.norm(data['I_vec'])*scale_factors['I'], 'Current', 'purple'),
        (data['tether_dir']*scale_factors['tether'], 'Tether Direction', 'orange'),
        (data['radial_dir']*scale_factors['radial'], 'Radial Direction', 'pink'),
        (data['velocity_dir']*scale_factors['velocity'], 'Velocity Direction', 'lightblue'),
        (data['F_lorentz']/np.linalg.norm(data['F_lorentz'])*scale_factors['F'], 'Lorentz Force', 'black')
    ]

    for vec, label, color in vectors:
        ax5.quiver(0, 0, 0, *vec, color=color, arrow_length_ratio=0.1, label=label)
    
    # 3D plot settings
    ax5.set_xlim([-base_scale, base_scale])
    ax5.set_ylim([-base_scale, base_scale])
    ax5.set_zlim([-base_scale, base_scale])
    ax5.set_xlabel('X (Radial)')
    ax5.set_ylabel('Y (Along-track)')
    ax5.set_zlabel('Z (Cross-track)')
    ax5.set_title(f'Orbit Time: {current_date.strftime("%Y-%m-%d %H:%M")}\n'
                 f'Altitude: {data["altitude"]:.1f} km\n'
                 f'Tether Angle: {TETHER_ANGLE_DEG}°\n'
                 f'Lorentz Force: {np.linalg.norm(data["F_lorentz"]):.2e} N\n'
                 f'Drag Force: {np.linalg.norm(data["F_drag"]):.2e} N\n'
                 f'Impact Rate: {data["impact_rate"]:.2e} impacts/s\n'
                 f'Total Impacts: {data["cumulative_impacts"]:.2e}\n'
                 f'Projected Area: {data["projected_area"]:.4f} m²\n'
                 f'Decay Rate: {decay_rate_m_per_day:.2f} m/day')
    ax5.legend(bbox_to_anchor=(1, 1), loc='upper left')
    ax5.view_init(elev=25, azim=35)
    ax5.set_box_aspect([1,1,1])
    
    # Update the vertical lines in all 2D plots
    pos_vline.set_xdata([current_date, current_date])
    force_vline.set_xdata([current_date, current_date])
    dv_vline.set_xdata([current_date, current_date])
    impact_vline.set_xdata([current_date, current_date])
    
    return pos_vline, force_vline, dv_vline, impact_vline

# Create animation
print("Creating animation...")
ani = FuncAnimation(fig, update, frames=len(frame_data),
                    interval=100, blit=False)

# Display in notebook
HTML(ani.to_jshtml())

Extras - old plots and functions

In [ ]:
# Altitude Profiles at Noon and Midnight (5° Off-Vertical Tether Orientation) - Validation Parameters
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
TETHER_ANGLE = np.radians(5)  # 5 degrees off vertical (radians)

# Single observation time (converted to Astropy Time)
obs_time = Time(datetime(2025, 3, 20, 12, 0, 0))  # Fixed universal time (Equinox)

# Find subsolar and anti-subsolar points at this time
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg  # Subsolar longitude
anti_subsolar_lon = (subsolar_lon + 180) % 360  # Opposite side

# Altitude range (100-1000 km)
altitudes_km = np.linspace(100, 1000, 50)
altitudes_m = altitudes_km * 1000 + R_EARTH

# Initialize arrays for storage
results = {
    'noon': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    },
    'midnight': {
        'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }
}


def get_position(alt_m, lon_deg):
    """Get position vector for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    return np.array([x, y, z]), np.array([x/1000, y/1000, z/1000])  # Return both m and km versions

# Calculate all parameters for both conditions
for alt_m, alt_km in zip(altitudes_m, altitudes_km):
    # Subsolar (noon) position
    r_m_noon, r_noon = get_position(alt_m, subsolar_lon)
    # Anti-subsolar (midnight) position
    r_m_midnight, r_midnight = get_position(alt_m, anti_subsolar_lon)
    
    # Velocity vectors (circular orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)  # Magnitude
    v_vec_noon = np.array([0, v_mag, 0])  # Simplified - would need proper transformation
    v_vec_midnight = np.array([0, -v_mag, 0])  # Opposite direction
    
    # Calculate radial and along-track directions
    radial_dir_noon = r_m_noon / np.linalg.norm(r_m_noon)
    v_perp_noon = v_vec_noon - np.dot(v_vec_noon, radial_dir_noon) * radial_dir_noon
    along_track_dir_noon = v_perp_noon / np.linalg.norm(v_perp_noon) if np.linalg.norm(v_perp_noon) > 0 else np.array([1,0,0])
    
    radial_dir_midnight = r_m_midnight / np.linalg.norm(r_m_midnight)
    v_perp_midnight = v_vec_midnight - np.dot(v_vec_midnight, radial_dir_midnight) * radial_dir_midnight
    along_track_dir_midnight = v_perp_midnight / np.linalg.norm(v_perp_midnight) if np.linalg.norm(v_perp_midnight) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir_noon = (np.cos(TETHER_ANGLE) * radial_dir_noon + 
                      np.sin(TETHER_ANGLE) * along_track_dir_noon)
    tether_dir_midnight = (np.cos(TETHER_ANGLE) * radial_dir_midnight + 
                         np.sin(TETHER_ANGLE) * along_track_dir_midnight)
    
    for condition, r_m, r, v_vec, tether_dir in [
        ('noon', r_m_noon, r_noon, v_vec_noon, tether_dir_noon),
        ('midnight', r_m_midnight, r_midnight, v_vec_midnight, tether_dir_midnight)
    ]:
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results[condition]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results[condition]['plasma_density'].append(n)
        results[condition]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results[condition]['atmos_density'].append(rho)
        
        # Velocity
        results[condition]['velocity'].append(v_mag)
        
        # Vemf (full tether length)
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results[condition]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        #I_mag = L * dIdl * np.sign(Vemf_proj)    # F prop L^2
        #I_mag = 1                                # TSS-1R
        I_mag = dIdl * np.sign(Vemf_proj)        # F prop L
        results[condition]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results[condition]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0])) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
        results[condition]['lorentz_force'].append(F_lorentz)

# NEW FUNCTIONALITY: Print values at specified altitude
def print_values_at_altitude(altitude_km):
    """Print all parameter values at a specified altitude"""
    # Find the closest altitude in our data
    idx = np.argmin(np.abs(altitudes_km - altitude_km))
    actual_altitude = altitudes_km[idx]
    
    print(f"\n{'='*80}")
    print(f"PARAMETER VALUES AT {actual_altitude:.1f} km ALTITUDE")
    print(f"{'='*80}")
    
    print(f"{'Parameter':<25} {'Noon':<15} {'Midnight':<15} {'Units':<10}")
    print(f"{'-'*80}")
    
    # Magnetic Field
    print(f"{'Magnetic Field':<25} {results['noon']['B_field'][idx]:<15.2e} {results['midnight']['B_field'][idx]:<15.2e} {'T':<10}")
    
    # Plasma Density
    print(f"{'Plasma Density':<25} {results['noon']['plasma_density'][idx]:<15.2e} {results['midnight']['plasma_density'][idx]:<15.2e} {'m⁻³':<10}")
    
    # Atmospheric Density
    print(f"{'Atmospheric Density':<25} {results['noon']['atmos_density'][idx]:<15.2e} {results['midnight']['atmos_density'][idx]:<15.2e} {'kg/m³':<10}")
    
    # Velocity
    print(f"{'Orbital Velocity':<25} {results['noon']['velocity'][idx]:<15.2f} {results['midnight']['velocity'][idx]:<15.2f} {'m/s':<10}")
    
    # Induced EMF
    print(f"{'Induced EMF (Vemf)':<25} {results['noon']['Vemf'][idx]:<15.2f} {results['midnight']['Vemf'][idx]:<15.2f} {'V':<10}")
    
    # Current
    print(f"{'Tether Current':<25} {results['noon']['current'][idx]:<15.2e} {results['midnight']['current'][idx]:<15.2e} {'A':<10}")
    
    # Drag Force
    print(f"{'Drag Force':<25} {results['noon']['drag_force'][idx]:<15.2e} {results['midnight']['drag_force'][idx]:<15.2e} {'N':<10}")
    
    # Lorentz Force
    print(f"{'Lorentz Force':<25} {results['noon']['lorentz_force'][idx]:<15.2e} {results['midnight']['lorentz_force'][idx]:<15.2e} {'N':<10}")
    
    # SZA
    print(f"{'Solar Zenith Angle':<25} {results['noon']['sza'][idx]:<15.1f} {results['midnight']['sza'][idx]:<15.1f} {'°':<10}")
    
    # Force Ratio
    ratio_noon = results['noon']['lorentz_force'][idx] / results['noon']['drag_force'][idx]
    ratio_midnight = results['midnight']['lorentz_force'][idx] / results['midnight']['drag_force'][idx]
    print(f"{'Lorentz/Drag Ratio':<25} {ratio_noon:<15.2f} {ratio_midnight:<15.2f} {'':<10}")
    
    # Net Force
    net_noon = results['noon']['lorentz_force'][idx] - results['noon']['drag_force'][idx]
    net_midnight = results['midnight']['lorentz_force'][idx] - results['midnight']['drag_force'][idx]
    print(f"{'Net Force (L-D)':<25} {net_noon:<15.2e} {net_midnight:<15.2e} {'N':<10}")
    
    print(f"{'='*80}")

# Print values at 700 km
print_values_at_altitude(700)

# Create the plots with drag vs Lorentz force comparison
plt.figure(figsize=(15, 20))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# 1. Magnetic Field
plt.subplot(4, 2, 1)
plt.plot(results['midnight']['B_field'], altitudes_km, label='Midnight (Anti-subsolar)')
plt.plot(results['noon']['B_field'], altitudes_km, label='Noon (Subsolar)')
plt.xlabel('Magnetic Field Strength (T)')
plt.ylabel('Altitude (km)')
plt.title('Magnetic Field vs Altitude')
plt.grid(True)
plt.legend()

# 2. Plasma Density
plt.subplot(4, 2, 2)
plt.semilogx(results['midnight']['plasma_density'], altitudes_km, label=f'Midnight (SZA={results["midnight"]["sza"][0]:.1f}°')
plt.semilogx(results['noon']['plasma_density'], altitudes_km, label=f'Noon (SZA={results["noon"]["sza"][0]:.1f}°')
plt.xlabel('Plasma Density (m$^{-3}$)')
plt.ylabel('Altitude (km)')
plt.title('Plasma Density vs Altitude\n(Day-Night Difference)')
plt.grid(True)
plt.legend()

# 3. Atmospheric Density
plt.subplot(4, 2, 3)
plt.semilogx(results['midnight']['atmos_density'], altitudes_km, label='Midnight')
plt.semilogx(results['noon']['atmos_density'], altitudes_km, label='Noon')
plt.xlabel('Atmospheric Density (kg/m$^3$)')
plt.ylabel('Altitude (km)')
plt.title('Atmospheric Density vs Altitude')
plt.grid(True)
plt.legend()

# 4. Orbital Velocity
plt.subplot(4, 2, 4)
plt.plot(results['midnight']['velocity'], altitudes_km, label='Midnight')
plt.plot(results['noon']['velocity'], altitudes_km, label='Noon')
plt.xlabel('Orbital Velocity (m/s)')
plt.ylabel('Altitude (km)')
plt.title('Circular Orbit Velocity vs Altitude')
plt.grid(True)
plt.legend()

# 5. Induced EMF (Vemf)
plt.subplot(4, 2, 5)
plt.plot(results['midnight']['Vemf'], altitudes_km, label='Midnight')
plt.plot(results['noon']['Vemf'], altitudes_km, label='Noon')
plt.xlabel('Induced EMF (V)')
plt.ylabel('Altitude (km)')
plt.title('Tether EMF vs Altitude')
plt.grid(True)
plt.legend()

# 6. Current
plt.subplot(4, 2, 6)
plt.semilogx(results['midnight']['current'], altitudes_km, label='Midnight')
plt.semilogx(results['noon']['current'], altitudes_km, label='Noon')
plt.xlabel('Current (A)')
plt.ylabel('Altitude (km)')
plt.title('Tether Current vs Altitude\n(Dominated by Plasma Density)')
plt.grid(True)
plt.legend()

# 7. Drag Force
plt.subplot(4, 2, 7)
plt.semilogx(results['midnight']['drag_force'], altitudes_km, label='Midnight')
plt.semilogx(results['noon']['drag_force'], altitudes_km, label='Noon')
plt.xlabel('Drag Force (N)')
plt.ylabel('Altitude (km)')
plt.title('Atmospheric Drag vs Altitude')
plt.grid(True)
plt.legend()

# 8. Lorentz Force
plt.subplot(4, 2, 8)
plt.semilogx(results['midnight']['lorentz_force'], altitudes_km, label='Midnight')
plt.semilogx(results['noon']['lorentz_force'], altitudes_km, label='Noon')
plt.xlabel('Lorentz Force (N)')
plt.ylabel('Altitude (km)')
plt.title('Electrodynamic Tether Force vs Altitude\n(Day-Night Difference)')
plt.grid(True)
plt.legend()

# New figure for force comparison
plt.figure(figsize=(12, 8))

# Drag vs Lorentz Force Comparison
plt.subplot(2, 2, 1)
plt.semilogx(results['noon']['drag_force'], altitudes_km, 'r-', label='Drag (Noon)')
plt.semilogx(results['noon']['lorentz_force'], altitudes_km, 'r--', label='Lorentz (Noon)')
plt.xlabel('Force (N)')
plt.ylabel('Altitude (km)')
plt.title('Drag vs Lorentz Force (Noon)')
plt.grid(True)
plt.legend()

plt.subplot(2, 2, 2)
plt.semilogx(results['midnight']['drag_force'], altitudes_km, 'b-', label='Drag (Midnight)')
plt.semilogx(results['midnight']['lorentz_force'], altitudes_km, 'b--', label='Lorentz (Midnight)')
plt.xlabel('Force (N)')
plt.ylabel('Altitude (km)')
plt.title('Drag vs Lorentz Force (Midnight)')
plt.grid(True)
plt.legend()

# Force Ratio
plt.subplot(2, 2, 3)
ratio_noon = np.array(results['noon']['lorentz_force']) / np.array(results['noon']['drag_force'])
ratio_midnight = np.array(results['midnight']['lorentz_force']) / np.array(results['midnight']['drag_force'])
plt.semilogx(ratio_noon, altitudes_km, 'g-', label='Noon (Lorentz/Drag)')
plt.semilogx(ratio_midnight, altitudes_km, 'm-', label='Midnight (Lorentz/Drag)')
plt.axvline(x=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Force Ratio (Lorentz/Drag)')
plt.ylabel('Altitude (km)')
plt.title('Lorentz to Drag Force Ratio')
plt.grid(True)
plt.legend()

# Force Difference
plt.subplot(2, 2, 4)
diff_noon = np.array(results['noon']['lorentz_force']) - np.array(results['noon']['drag_force'])
diff_midnight = np.array(results['midnight']['lorentz_force']) - np.array(results['midnight']['drag_force'])
plt.plot(diff_noon, altitudes_km, 'g-', label='Noon (Lorentz - Drag)')
plt.plot(diff_midnight, altitudes_km, 'm-', label='Midnight (Lorentz - Drag)')
plt.axvline(x=0, color='k', linestyle='--', alpha=0.5, label='Zero Net Force')
plt.xlabel('Net Force (N)')
#plt.xscale('log')
plt.ylabel('Altitude (km)')
plt.title('Net Electrodynamic Force\n(Lorentz - Drag)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Longitude Profiles at 300km and 700km (5° Off-Vertical Tether Orientation)

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun

# Constants
R_EARTH = 6378e3  # Earth radius (m)
M_EARTH = 5.972e24  # Earth mass (kg)
G = 6.674e-11  # Gravitational constant
MU_EARTH = G * M_EARTH  # Earth's gravitational parameter
e = 1.60217663e-19  # elementary charge (C)
Me = 9.1093837e-31  # electron mass (kg)
Mi = 1.67262192e-27  # proton mass (kg)
TETHER_ANGLE = np.radians(5)  # 5 degrees off vertical (radians)


# Observation time and sun position
obs_time = Time(datetime(2021, 10, 13, 12, 0, 0))  # Noon UTC
sun = get_sun(obs_time)
subsolar_lon = sun.ra.deg

# Fixed altitudes
altitudes = [300, 700]  # km
altitudes_m = [alt*1000 + R_EARTH for alt in altitudes]

# Longitude range (0-360° in 5° steps)
longitudes = np.linspace(0, 360, 73)  # 5° resolution

# Initialize storage without torque data
results = {
    '300km': {
        'longitude': [], 'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    },
    '700km': {
        'longitude': [], 'B_field': [], 'plasma_density': [], 'atmos_density': [],
        'velocity': [], 'Vemf': [], 'current': [], 
        'drag_force': [], 'lorentz_force': [], 'sza': []
    }
}


def get_position_and_tether_dir(alt_m, lon_deg):
    """Get position vector and tether direction for given altitude and longitude"""
    lat = 0  # Equatorial orbit
    earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat, height=alt_m-R_EARTH)
    x = earth_location.x.value
    y = earth_location.y.value
    z = earth_location.z.value
    
    # Position vectors (m and km)
    r_m = np.array([x, y, z])
    r_km = r_m / 1000
    
    # Calculate directions
    radial_dir = r_m / np.linalg.norm(r_m)
    
    # Velocity vector (tangential to orbit)
    v_mag = np.sqrt(G*M_EARTH/alt_m)
    v_vec = np.array([-v_mag*np.sin(np.radians(lon_deg)), 
                      v_mag*np.cos(np.radians(lon_deg)), 
                      0])
    
    # Calculate along-track direction
    v_perp = v_vec - np.dot(v_vec, radial_dir) * radial_dir
    along_track_dir = v_perp / np.linalg.norm(v_perp) if np.linalg.norm(v_perp) > 0 else np.array([1,0,0])
    
    # Tether direction (5 degrees off radial)
    tether_dir = np.cos(TETHER_ANGLE) * radial_dir + np.sin(TETHER_ANGLE) * along_track_dir
    
    return r_m, r_km, tether_dir, v_vec

# Calculate parameters for both altitudes
for alt_km, alt_m in zip(altitudes, altitudes_m):
    key = f"{alt_km}km"
    
    for lon in longitudes:
        r_m, r_km, tether_dir, v_vec = get_position_and_tether_dir(alt_m, lon)
        v_mag = np.linalg.norm(v_vec)
        
        # Magnetic field
        B = igrf_cartesian(obs_time.datetime, r_km) * 1e-9  # Convert nT to T
        B_mag = np.linalg.norm(B)
        results[key]['B_field'].append(B_mag)
        
        # Plasma density and SZA
        n, sza = calculate_plasma_density(r_m, obs_time.datetime)
        results[key]['plasma_density'].append(n)
        results[key]['sza'].append(sza)
        
        # Atmospheric density
        rho, _ = atmospheric_density_nrlmsise00(r_m, obs_time.datetime)
        results[key]['atmos_density'].append(rho)
        
        # Velocity
        results[key]['velocity'].append(v_mag)
        
        # Vemf
        v_cross_B = np.cross(v_vec, B[:,0])
        Vemf_total = L * v_cross_B
        Vemf_proj = np.dot(Vemf_total, tether_dir)  # Projected along tether
        results[key]['Vemf'].append(np.linalg.norm(Vemf_proj))
        
        # Current
        dIdl = I_func(r_m, obs_time.datetime, abs(Vemf_proj)) # Current/length
        I_mag = dIdl * np.sign(Vemf_proj)
        results[key]['current'].append(abs(I_mag))
        
        # Drag force
        tether_projected_area = L * w * np.abs(np.cos(TETHER_ANGLE))
        total_area = Area1 + tether_projected_area
        F_drag_mag = 0.5 * Cd1 * total_area * rho * v_mag**2
        results[key]['drag_force'].append(F_drag_mag)
        
        # Lorentz force
        F_lorentz = L * np.linalg.norm(np.cross(I_mag * tether_dir, B[:,0])) #* 0.5  # Factor of 0.5 from integral of I(L) x B * dl = IB * I^2/s
        results[key]['lorentz_force'].append(F_lorentz)
        
        # Longitude
        results[key]['longitude'].append(lon)

# Create the plots with drag vs Lorentz force comparison
plt.figure(figsize=(15, 20))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# Vertical lines for subsolar/anti-subsolar points
subsol_line = {'ls': '--', 'color': 'gold', 'alpha': 0.7, 'label': 'Subsolar'}
antisol_line = {'ls': '--', 'color': 'navy', 'alpha': 0.7, 'label': 'Anti-subsolar'}

# 1. Magnetic Field
plt.subplot(4, 2, 1)
plt.plot(results['300km']['longitude'], results['300km']['B_field'], label='300 km')
plt.plot(results['700km']['longitude'], results['700km']['B_field'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Magnetic Field (T)')
plt.title('B-field vs Longitude (Tether 5° off vertical)')
plt.grid(True)
plt.legend()

# 2. Plasma Density
plt.subplot(4, 2, 2)
plt.semilogy(results['300km']['longitude'], results['300km']['plasma_density'], label='300 km')
plt.semilogy(results['700km']['longitude'], results['700km']['plasma_density'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Plasma Density (m$^{-3}$)')
plt.title('Plasma Density vs Longitude')
plt.grid(True)
plt.legend()

# 3. Atmospheric Density
plt.subplot(4, 2, 3)
plt.semilogy(results['300km']['longitude'], results['300km']['atmos_density'], label='300 km')
plt.semilogy(results['700km']['longitude'], results['700km']['atmos_density'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Atmospheric Density (kg/m$^3$)')
plt.title('Atmospheric Density vs Longitude')
plt.grid(True)
plt.legend()

# 4. Orbital Velocity
plt.subplot(4, 2, 4)
plt.plot(results['300km']['longitude'], results['300km']['velocity'], label='300 km')
plt.plot(results['700km']['longitude'], results['700km']['velocity'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Velocity (m/s)')
plt.title('Orbital Velocity vs Longitude')
plt.grid(True)
plt.legend()

# 5. Induced EMF
plt.subplot(4, 2, 5)
plt.plot(results['300km']['longitude'], results['300km']['Vemf'], label='300 km')
plt.plot(results['700km']['longitude'], results['700km']['Vemf'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('EMF (V)')
plt.title('Tether EMF vs Longitude (Tether 5° off vertical)')
plt.grid(True)
plt.legend()

# 6. Current
plt.subplot(4, 2, 6)
plt.semilogy(results['300km']['longitude'], results['300km']['current'], label='300 km')
plt.semilogy(results['700km']['longitude'], results['700km']['current'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Current (A)')
plt.title('Tether Current vs Longitude')
plt.grid(True)
plt.legend()

# 7. Drag Force
plt.subplot(4, 2, 7)
plt.semilogy(results['300km']['longitude'], results['300km']['drag_force'], label='300 km')
plt.semilogy(results['700km']['longitude'], results['700km']['drag_force'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Drag Force (N)')
plt.title('Atmospheric Drag vs Longitude')
plt.grid(True)
plt.legend()

# 8. Lorentz Force
plt.subplot(4, 2, 8)
plt.semilogy(results['300km']['longitude'], results['300km']['lorentz_force'], label='300 km')
plt.semilogy(results['700km']['longitude'], results['700km']['lorentz_force'], label='700 km')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Lorentz Force (N)')
plt.title('Electrodynamic Tether Force vs Longitude (Tether 5° off vertical)')
plt.grid(True)
plt.legend()

# New figure for force comparison
plt.figure(figsize=(15, 10))

# Drag vs Lorentz Force Comparison for 300km
plt.subplot(2, 2, 1)
plt.semilogy(results['300km']['longitude'], results['300km']['drag_force'], 'b-', label='Drag (300km)')
plt.semilogy(results['300km']['longitude'], results['300km']['lorentz_force'], 'b--', label='Lorentz (300km)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Force (N)')
plt.title('Drag vs Lorentz Force (300km)')
plt.grid(True)
plt.legend()

# Drag vs Lorentz Force Comparison for 600km
plt.subplot(2, 2, 2)
plt.semilogy(results['700km']['longitude'], results['700km']['drag_force'], 'r-', label='Drag (700km)')
plt.semilogy(results['700km']['longitude'], results['700km']['lorentz_force'], 'r--', label='Lorentz (700km)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.xlabel('Longitude (°)')
plt.ylabel('Force (N)')
plt.title('Drag vs Lorentz Force (700km)')
plt.grid(True)
plt.legend()

# Force Ratio
plt.subplot(2, 2, 3)
ratio_300km = np.array(results['300km']['lorentz_force']) / np.array(results['300km']['drag_force'])
ratio_600km = np.array(results['700km']['lorentz_force']) / np.array(results['700km']['drag_force'])
plt.semilogy(results['300km']['longitude'], ratio_300km, 'g-', label='300km (Lorentz/Drag)')
plt.semilogy(results['700km']['longitude'], ratio_600km, 'm-', label='700km (Lorentz/Drag)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Equal Force (Ratio=1)')
plt.xlabel('Longitude (°)')
plt.ylabel('Force Ratio (Lorentz/Drag)')
plt.title('Lorentz to Drag Force Ratio')
plt.grid(True)
plt.legend()

# Force Difference
plt.subplot(2, 2, 4)
diff_300km = np.array(results['300km']['lorentz_force']) - np.array(results['300km']['drag_force'])
diff_600km = np.array(results['700km']['lorentz_force']) - np.array(results['700km']['drag_force'])
plt.plot(results['300km']['longitude'], diff_300km, 'g-', label='300km (Lorentz - Drag)')
plt.plot(results['700km']['longitude'], diff_600km, 'm-', label='700km (Lorentz - Drag)')
plt.axvline(subsolar_lon, **subsol_line)
plt.axvline((subsolar_lon+180)%360, **antisol_line)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5, label='Zero Net Force')
plt.xlabel('Longitude (°)')
plt.ylabel('Net Force (N)')
plt.title('Net Electrodynamic Force\n(Lorentz - Drag)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Old janky plasma density function

def calculate_plasma_density(sat_position_earthmj2000eq, observation_time):
    """
    Calculate plasma density at satellite position using Chapman function with solar zenith angle.
    
    Parameters:
    sat_position_earthmj2000eq : array-like (x, y, z) in meters in EarthMJ2000Eq frame
    observation_time : datetime object
    
    Returns:
    plasma_density : float in particles/m^3
    """
    # Constants
    H = 165e3         # Scale height in meters (typical for ionosphere)      Source: https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2018ja026280 
    N0 = 1e12         # Peak plasma density in particles/m^3 (dayside)       Sources: https://www.ngdc.noaa.gov/stp/space-weather/online-publications/miscellaneous/afrl_publications/handbook_1985/Chptr09.pdf, https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2023RS007658 
    N0_night = 1e11   # Base plasma density for nightside in particles/m^3   


    # Convert position to ITRS (Earth-fixed) coordinates
    from astropy.time import Time                              # To convert time to astropy-compatible format
    observation_time_astropy = Time(observation_time)
    
    # Create cartesian representation of the position
    from astropy.coordinates import CartesianRepresentation    # To record cartesian vector for astropy
    import astropy.units as u                                  # Import units
    cart_rep = CartesianRepresentation(
        x=sat_position_earthmj2000eq[0] * u.m,                 # Position in meters
        y=sat_position_earthmj2000eq[1] * u.m,
        z=sat_position_earthmj2000eq[2] * u.m
    )
    
    # Create GCRS coordinate
    from astropy.coordinates import GCRS                     # Geocentric Celestial Reference System
    gcrs_coord = GCRS(cart_rep, obstime=observation_time_astropy)
    
    # Transform to ITRS coordinate
    from astropy.coordinates import ITRS                     # International Terrestrial Reference System (Earth-fixed coordinate system)
    itrs_coord = gcrs_coord.transform_to(ITRS(obstime=observation_time_astropy))

    # Get geodetic coordinates (latitude, longitude, height)
    from astropy.coordinates import EarthLocation            # Represents a location on Earth's surface using geodetic or geocentric coordinates
    earth_location = EarthLocation.from_geocentric(itrs_coord.x, itrs_coord.y, itrs_coord.z)
    height = earth_location.height.value 

    # Why is the previous step necessary instead of just calculating the altitude? Dunno

    # Calculate solar zenith angle (Angle between Earth->Sun and Earth->satellite vectors)
    
    # Function to get suns position in the sky
    from astropy.coordinates import get_sun          
    
    # Get Sun position in ITRS (Earth-fixed frame)
    sun_itrs = get_sun(observation_time_astropy).transform_to(ITRS(obstime=observation_time_astropy))

    # Find Sun and Satellite vectors in ITRS
    sat_vec = np.array([itrs_coord.cartesian.x.value,
                        itrs_coord.cartesian.y.value,
                        itrs_coord.cartesian.z.value])
    sun_vec = np.array([sun_itrs.cartesian.x.value,
                        sun_itrs.cartesian.y.value,
                        sun_itrs.cartesian.z.value])

    # Compute the angle between satellite and Sun
    dot_product = np.dot(sat_vec, sun_vec)
    norm_sat = np.linalg.norm(sat_vec)
    norm_sun = np.linalg.norm(sun_vec)
    cos_theta = dot_product / (norm_sat * norm_sun)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    solar_zenith = np.degrees(np.arccos(cos_theta))  # in degrees

    h_day_peak = 2.75e5    # Source: https://www.ngdc.noaa.gov/stp/space-weather/online-publications/miscellaneous/afrl_publications/handbook_1985/Chptr09.pdf 
    h_night_peak = 3.5e5
    
    # Calculate Chapman function for dayside (0° < SZA < 90°)
    
    if solar_zenith < 75: #90:
        # Dayside Chapman function: N = N0 * exp(1 - z - sec(χ) * exp(-z))     Source: https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1002/2014JA020665
        # where z = ((r - R_earth) - h_peak_production)/H

        z = (height - h_day_peak) / H
        sec_chi = 1 / np.cos(np.radians(solar_zenith))
        sec_chi = min(sec_chi, 1 / np.cos(np.radians(77)))         # Keeps plasma density from going to zero at dawn and dusk

        plasma_density = N0 * np.exp(1 - z - sec_chi * np.exp(-z)) # Chapman function
        plasma_density = np.clip(plasma_density, 1e9, 1e13)


    else:
        z = (height - h_night_peak) / H
        sec_chi = np.abs(1 / np.cos(np.radians(180)))                     # Keep fixed so no longitude dependency at night
        plasma_density = N0_night * np.exp(1 - z - sec_chi * np.exp(-z))  # Chapman function with nightime peak density
        plasma_density = np.clip(plasma_density, 1e9, 1e13)

    return plasma_density, solar_zenith


In [ ]:
# Test cell to verify the new plasma model
# Run this after replacing the original calculate_plasma_density function

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm

# Define test parameters
R_EARTH = 6378e3  # Earth radius in meters
altitudes_km = np.linspace(100, 1000, 30)  # Altitude range
longitudes = np.linspace(0, 360, 37)  # Longitude range
latitudes = [0, 45, -45]  # Test at equator, mid-north, mid-south

# Test time (noon UT)
test_time = datetime(2025, 3, 20, 12, 0, 0)

# Create figure for comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Plasma Density Comparison: IRI Model vs Original Chapman Model\n'
             f'Test Date: {test_time.strftime("%Y-%m-%d %H:%M UTC")}',
             fontsize=14, fontweight='bold')

# Store original function for comparison (if still available)
def calculate_plasma_density_original(sat_position_earthmj2000eq, observation_time):
    """Original Chapman model for comparison"""
    H = 165e3
    N0 = 1e12
    N0_night = 1e11
    h_day_peak = 275e3
    h_night_peak = 350e3
    
    # Simple position conversion for test
    r = np.linalg.norm(sat_position_earthmj2000eq)
    height = r - R_EARTH
    
    # Simplified SZA calculation
    from astropy.time import Time
    from astropy.coordinates import EarthLocation, ITRS, GCRS, CartesianRepresentation, get_sun
    import astropy.units as u
    
    observation_time_astropy = Time(observation_time)
    cart_rep = CartesianRepresentation(
        x=sat_position_earthmj2000eq[0] * u.m,
        y=sat_position_earthmj2000eq[1] * u.m,
        z=sat_position_earthmj2000eq[2] * u.m
    )
    gcrs_coord = GCRS(cart_rep, obstime=observation_time_astropy)
    itrs_coord = gcrs_coord.transform_to(ITRS(obstime=observation_time_astropy))
    sun_itrs = get_sun(observation_time_astropy).transform_to(ITRS(obstime=observation_time_astropy))
    
    sat_vec = np.array([itrs_coord.cartesian.x.value, itrs_coord.cartesian.y.value, itrs_coord.cartesian.z.value])
    sun_vec = np.array([sun_itrs.cartesian.x.value, sun_itrs.cartesian.y.value, sun_itrs.cartesian.z.value])
    
    dot_product = np.dot(sat_vec, sun_vec)
    cos_theta = dot_product / (np.linalg.norm(sat_vec) * np.linalg.norm(sun_vec))
    solar_zenith = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
    
    if solar_zenith < 75:
        z = (height - h_day_peak) / H
        sec_chi = 1 / np.cos(np.radians(min(solar_zenith, 77)))
        plasma_density = N0 * np.exp(1 - z - sec_chi * np.exp(-z))
    else:
        z = (height - h_night_peak) / H
        sec_chi = np.abs(1 / np.cos(np.radians(180)))
        plasma_density = N0_night * np.exp(1 - z - sec_chi * np.exp(-z))
    
    return np.clip(plasma_density, 1e9, 1e13), solar_zenith

# Test at different latitudes
for idx, lat in enumerate(latitudes):
    # Create position function for given latitude and longitude
    def get_position(alt_m, lon_deg, lat_deg=lat):
        """Get position vector for given altitude and longitude at fixed latitude"""
        from astropy.coordinates import EarthLocation
        earth_location = EarthLocation.from_geodetic(lon=lon_deg, lat=lat_deg, height=alt_m-R_EARTH)
        return np.array([earth_location.x.value, earth_location.y.value, earth_location.z.value])
    
    # Initialize arrays
    plasma_iri = np.zeros((len(altitudes_km), len(longitudes)))
    plasma_original = np.zeros((len(altitudes_km), len(longitudes)))
    
    # Calculate plasma density
    print(f"Calculating for latitude {lat}°...")
    for i, alt_km in enumerate(tqdm(altitudes_km)):
        alt_m = alt_km * 1000 + R_EARTH
        for j, lon in enumerate(longitudes):
            pos = get_position(alt_m, lon)
            
            # New IRI model
            try:
                plasma_iri[i, j], _ = calculate_plasma_density(pos, test_time)
            except Exception as e:
                print(f"Error at lat={lat}, lon={lon}, alt={alt_km}km: {e}")
                plasma_iri[i, j] = np.nan
            
            # Original model for comparison
            plasma_original[i, j], _ = calculate_plasma_density_original(pos, test_time)
    
    # Plot results
    ax1 = axes[0, idx]
    im1 = ax1.pcolormesh(longitudes, altitudes_km, np.log10(plasma_iri), 
                         shading='auto', cmap='viridis')
    ax1.set_xlabel('Longitude (°)')
    ax1.set_ylabel('Altitude (km)')
    ax1.set_title(f'IRI Model - Latitude {lat}°\nlog10(Ne) [m⁻³]')
    plt.colorbar(im1, ax=ax1)
    
    ax2 = axes[1, idx]
    im2 = ax2.pcolormesh(longitudes, altitudes_km, np.log10(plasma_original), 
                         shading='auto', cmap='viridis')
    ax2.set_xlabel('Longitude (°)')
    ax2.set_ylabel('Altitude (km)')
    ax2.set_title(f'Original Chapman Model - Latitude {lat}°\nlog10(Ne) [m⁻³]')
    plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

# Additional verification at specific point
print("\n" + "="*60)
print("VERIFICATION AT SPECIFIC POINT")
print("="*60)

# Test at 500 km altitude, equator, 0° longitude
test_alt_km = 500
test_lon = 0
test_lat = 0
test_time = datetime(2025, 3, 20, 12, 0, 0)

from astropy.coordinates import EarthLocation
alt_m = test_alt_km * 1000 + R_EARTH
earth_location = EarthLocation.from_geodetic(lon=test_lon, lat=test_lat, height=alt_m-R_EARTH)
test_pos = np.array([earth_location.x.value, earth_location.y.value, earth_location.z.value])

# Original model
density_original, sza_original = calculate_plasma_density_original(test_pos, test_time)

# New IRI model
density_iri, sza_iri = calculate_plasma_density(test_pos, test_time)

print(f"Test Point: Altitude={test_alt_km} km, Latitude={test_lat}°, Longitude={test_lon}°")
print(f"Time: {test_time}")
print(f"\nOriginal Chapman Model:")
print(f"  Plasma Density: {density_original:.2e} m⁻³")
print(f"  Solar Zenith Angle: {sza_original:.1f}°")
print(f"\nNew IRI Model:")
print(f"  Plasma Density: {density_iri:.2e} m⁻³")
print(f"  Solar Zenith Angle: {sza_iri:.1f}°")
print(f"\nRatio (IRI/Original): {density_iri/density_original:.2f}")

# Check if the function works within the original analysis pipeline
print("\n" + "="*60)
print("INTEGRATION TEST WITH ORIGINAL PIPELINE")
print("="*60)

# Test if the function returns correct format for I_func
from datetime import datetime
test_time = datetime(2025, 3, 20, 12, 0, 0)
density, sza = calculate_plasma_density(test_pos, test_time)
print(f"Function returns: density={density:.2e} m⁻³, sza={sza:.1f}°")
print(f"Return type: {type(density)}, {type(sza)}")
print(f"✓ Function works with I_func (expects plasma density in m⁻³)")

# Test multiple points quickly
print("\nTesting multiple altitudes at equator, 0° longitude:")
for alt in [200, 400, 600, 800, 1000]:
    alt_m = alt * 1000 + R_EARTH
    earth_location = EarthLocation.from_geodetic(lon=0, lat=0, height=alt_m-R_EARTH)
    pos = np.array([earth_location.x.value, earth_location.y.value, earth_location.z.value])
    density, sza = calculate_plasma_density(pos, test_time)
    print(f"  {alt} km: {density:.2e} m⁻³ (SZA={sza:.1f}°)")